# CesiCDP · Static CVRPTW mathematical modeling
## Deliverable 1 — capacity, hard service-start windows and visual validation

The selected scope is one depot, at most K identical-capacity vehicles, mandatory unsplit customers, static travel, customer service durations, hard service-start windows, early waiting, depot opening/closing, and minimum total traveled distance under the bundled reference convention.

This notebook develops routing and temporal equations, then validates supplied Solomon and Gehring–Homberger reference routes. Sections 12–15 add Algorithm 1, **ALNS + VND**: its complete source code (this notebook is its authoritative source), a live demonstration and the frozen held-out benchmark results. ILS and HGS are not implemented yet.

## Notebook navigation / deliverable checklist

- [x] Context and problem scope
- [x] Formal data, notation, and assumptions
- [x] Decision variables and domains
- [x] Objective function and full ILP/MILP formulation
- [x] All feasibility constraints, including subtour elimination
- [x] NP-hardness proof (TSP reduction)
- [x] Illustrative model and constraint visualization
- [x] Synthetic, parameterized data demonstrations
- [x] Real CVRPTW instances and independent reference validation
- [x] Mathematical-model sizing and limitations
- [x] Algorithm 1 — ALNS + VND: source code, live demonstration, held-out results (Sections 12–15)
- [ ] Algorithms 2–3 (ILS, HGS) and a PyVRP baseline (pending approval)
- [x] Bibliographic references
- [ ] Tutor review, feedback, and correction log (to be completed by the team)

**Storytelling principle:** each section asks a concrete question, introduces the matching model component, and visually demonstrates why it matters.

The temporal formulation and static CVRPTW reference demonstrations appear in Sections 4.5 and 10A. All reported reference costs remain external results.

## 0 · Execution, configuration and evidence

Install `numpy pandas matplotlib vrplib` (and `nbformat nbclient ipykernel` for automated execution). Run from the repository root, or set `ECOROUTE_PROJECT_ROOT`. Dataset root defaults to `data/benchmarks/vrptw/`; override with `ECOROUTE_DATA_ROOT`. The default modeling run executes synthetic checks and marks collection-dependent work **NOT RUN** when files are missing. Set `ECOROUTE_STRICT_DATA=1` to fail on any missing pair. Present but malformed data or incorrect reference costs always fail, in either mode.

Download separately with `python scripts/download_vrptw_benchmarks.py`. Run All never downloads or changes source data. Saved outputs record one execution, not evidence of a fresh rerun on the reader's machine. Live UTC time, package versions, source manifest and per-file/collection hashes appear below. The synthetic seed is `SYNTHETIC.seed`; reference checking uses no randomness.


In [ ]:
from __future__ import annotations

# Standard library dependencies: portable file paths, immutable configurations,
# annotations, regex parsing, and integer arithmetic.
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import math
import re
import warnings

# Scientific Python libraries are used for data analysis and plotting ONLY.
# No optimization solver or metaheuristic package is imported.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from IPython.display import display

try:
    import vrplib  # Preferred, official parser used in the later project phases.
    VRPLIB_AVAILABLE = True
except ImportError:
    vrplib = None
    VRPLIB_AVAILABLE = False

# Sensible display settings affect chart readability, never model constraints.
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 105,
    "axes.grid": True,
    "grid.alpha": 0.18,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
pd.set_option("display.max_columns", 20)

print("vrplib installed:", VRPLIB_AVAILABLE)
print("NumPy:", np.__version__, "· Matplotlib:", plt.matplotlib.__version__)

# Reproducible, configurable roots. No network acquisition during Run All.
import os, sys, json, platform, hashlib
from datetime import datetime, timezone
PROJECT_ROOT = Path(os.environ.get("ECOROUTE_PROJECT_ROOT", Path.cwd())).resolve()
if not (PROJECT_ROOT / "scripts" / "cvrptw_model.py").is_file():
    raise FileNotFoundError("Run from repository root or set ECOROUTE_PROJECT_ROOT")
sys.path.insert(0, str(PROJECT_ROOT))
DATA_ROOT = Path(os.environ.get("ECOROUTE_DATA_ROOT", PROJECT_ROOT / "data" / "benchmarks" / "vrptw")).resolve()
STRICT_DATA = os.environ.get("ECOROUTE_STRICT_DATA", "0") == "1"
print("Fresh execution UTC:", datetime.now(timezone.utc).isoformat())
print("Dataset root:", DATA_ROOT, "strict:", STRICT_DATA)


## 1 · From a delivery network to a graph

Let $G=(V,A)$ be a directed graph, where:

- $V=\{0,1,\ldots,n\}$ contains depot $0$ and customers.
- $A=\{(i,j):i,j\in V,\;i\ne j\}$ contains inter-location arcs.
- $F=\{1,\ldots,K\}$ contains at most $K$ available vehicles with capacity $Q$.
- Customer $i$ has demand $q_i$, service duration $s_i$, and service-start window $[a_i,b_i]$.
- $d_{ij}$ is the static travel distance between nodes; coordinates may be used to derive it.

Every customer is served exactly once, each used route starts and ends at the depot, capacity and time windows hold, and early arrival may wait. The objective is total route distance.

Distances and travel durations are separate configurable static matrices (equal under the selected native benchmark convention). Vehicles may choose a departure anywhere within the **same depot window**; no truck-specific departure windows are introduced. Waiting, travel duration and utilization are reported metrics, not objective terms.


### 1.1 · The difference between a parameter and a hard-coded decision

The *formula* $\sum_{i\in C}q_i y_{ik}\le Qz_k$ never changes. Only the supplied instance values $q_i$, $Q$, and $K$ change. A small demonstration with chosen numbers is not a restriction on the general model.

**Editable synthetic-data configuration:** the following values are illustrative defaults only. Change them to visualize different sizes, capacities, fleet limits, customer layouts, and demand distributions. There is **no optimizer** in the generator.

In [ ]:
@dataclass(frozen=True)
class SyntheticConfig:
    # Every attribute below belongs to the input configuration, not to the ILP.
    n_customers: int = 42
    max_vehicles: int = 8
    vehicle_capacity: float = 35.0
    min_demand: int = 1
    max_demand: int = 9
    coordinate_max: float = 100.0
    depot_xy: tuple[float, float] = (50.0, 50.0)
    layout: str = "uniform"  # Other supported layout: "clustered".
    seed: int = 2026
    distance_policy: str = "raw"  # "raw", "nearest_integer", or "ceil".

# Change only this cell to change the synthetic *instance* across its plots.
SYNTHETIC = SyntheticConfig()

# A separately configurable small teaching example for route feasibility:
# these are illustrative example inputs, not limits imposed by the model.
EXAMPLE_COORDS = np.array([
    [50, 50], [14, 70], [21, 42], [35, 16],
    [75, 76], [86, 45], [62, 20],
], dtype=float)
EXAMPLE_DEMANDS = np.array([0, 6, 5, 4, 7, 3, 2], dtype=float)
EXAMPLE_CAPACITY = 16.0
EXAMPLE_MAX_VEHICLES = 3
EXAMPLE_ROUTES = [[1, 2, 3], [4, 5, 6]]  # Hand-written, NOT computed.

### 1.2 · A reusable, validated instance container

The instance container checks **data consistency**, not optimality. A minimal fleet lower bound is $\lceil (\sum_i q_i)/Q\rceil$; the inequality $\sum_i q_i\le KQ$ is necessary but not sufficient for packing customers into $K$ trucks. Example: three customers with demand 6 each, two trucks of capacity 10: total demand 18 is below 20, but the customers cannot fit in two unsplittable routes.

For convenience in code, every instance is internally **reindexed so the depot is location 0**. A mapping is retained for source dataset identifiers.

This capacity-only container is a **teaching fixture** for routing equations and plots. The only solver/benchmark input is `scripts.cvrptw_model.CVRPTWInstance`; shared interfaces reject this legacy container. Full time-window checks use `check_cvrptw` in §10A.


In [ ]:
@dataclass
class CVRPInstance:
    name: str
    demands: np.ndarray          # One demand per node; depot demand must be 0.
    capacity: float               # Shared capacity of all K vehicles.
    max_vehicles: int             # Fleet upper bound; can differ by instance.
    distances: np.ndarray         # Matrix [node, node], supplied or generated.
    coordinates: np.ndarray | None = None
    original_ids: np.ndarray | None = None
    source: str = "synthetic"
    distance_policy: str = "raw"

    @property
    def n_customers(self) -> int:
        return len(self.demands) - 1

    @property
    def nodes(self) -> np.ndarray:
        return np.arange(len(self.demands))

    def check_input(self) -> list[str]:
        """Return problems, instead of silently assuming the data are valid."""
        problems = []
        q = np.asarray(self.demands)
        if q.ndim != 1 or q.size < 2:
            return ["Demand must be a vector containing depot plus customers"]
        m = q.size
        D = np.asarray(self.distances)
        if m < 2:
            problems.append("At least one customer is required for this demonstration.")
        if not isinstance(self.max_vehicles, (int, np.integer)) or self.max_vehicles < 1:
            problems.append("Fleet size must be a positive integer.")
        if not np.isfinite(self.capacity) or self.capacity <= 0:
            problems.append("Vehicle capacity must be strictly positive.")
        if len(np.asarray(self.demands).shape) != 1:
            problems.append("Demand vector must be one-dimensional.")
        elif not np.all(np.isfinite(self.demands)) or np.any(self.demands < 0):
            problems.append("Demands must be finite and nonnegative.")
        if len(self.demands) and not np.isclose(self.demands[0], 0):
            problems.append("The depot must have zero delivery demand.")
        if D.shape != (m, m):
            problems.append(f"Cost matrix has shape {D.shape}, expected {(m,m)}.")
        elif not np.all(np.isfinite(D)) or np.any(D < 0):
            problems.append("Distances must be finite and nonnegative.")
        if self.coordinates is not None and np.asarray(self.coordinates).shape != (m, 2):
            problems.append("Coordinates must have shape (number_of_nodes, 2).")
        if self.original_ids is not None and np.asarray(self.original_ids).shape != (m,):
            problems.append("Original ID mapping length does not match the instance.")
        if not problems and np.any(self.demands[1:] > self.capacity):
            problems.append("At least one customer demand exceeds one truck's capacity.")
        return problems


def euclidean_cost_matrix(coords: np.ndarray, policy: str = "raw") -> np.ndarray:
    """Compute pairwise 2-D straight-line distances; no routing takes place.

    'nearest_integer' intentionally uses floor(d + 0.5), rather than np.round:
    np.round uses ties-to-even, whereas many benchmark conventions use
    nearest-integer rounding. Always verify the convention per instance set.
    """
    coords = np.asarray(coords, dtype=float)
    delta = coords[:, None, :] - coords[None, :, :]
    D = np.sqrt(np.sum(delta * delta, axis=2))
    if policy == "raw":
        pass
    elif policy == "nearest_integer":
        D = np.floor(D + 0.5)
    elif policy == "ceil":
        D = np.ceil(D)
    else:
        raise ValueError("distance policy must be raw, nearest_integer, or ceil")
    np.fill_diagonal(D, 0.0)
    return D


def generate_synthetic(cfg: SyntheticConfig) -> CVRPInstance:
    """Generate input only. Never creates, evaluates, or optimizes routes."""
    if cfg.n_customers < 1 or cfg.min_demand < 0 or cfg.min_demand > cfg.max_demand:
        raise ValueError("Invalid customer count or demand range")
    rng = np.random.default_rng(cfg.seed)
    n = cfg.n_customers
    if cfg.layout == "uniform":
        customer_coords = rng.uniform(0, cfg.coordinate_max, size=(n, 2))
    elif cfg.layout == "clustered":
        centers = rng.uniform(0.15, 0.85, size=(3, 2)) * cfg.coordinate_max
        owners = rng.integers(0, len(centers), size=n)
        customer_coords = centers[owners] + rng.normal(
            0, 0.075 * cfg.coordinate_max, size=(n, 2)
        )
        customer_coords = np.clip(customer_coords, 0, cfg.coordinate_max)
    else:
        raise ValueError("Unknown layout. Use uniform or clustered.")
    coords = np.vstack([np.array(cfg.depot_xy), customer_coords])
    demands = np.r_[0, rng.integers(cfg.min_demand, cfg.max_demand + 1, n)]
    instance = CVRPInstance(
        name=f"Synthetic ({cfg.layout}, n={n}, seed={cfg.seed})",
        demands=demands.astype(float),
        capacity=float(cfg.vehicle_capacity),
        max_vehicles=int(cfg.max_vehicles),
        distances=euclidean_cost_matrix(coords, cfg.distance_policy),
        coordinates=coords,
        original_ids=np.arange(n + 1),
        source="generated data",
        distance_policy=cfg.distance_policy,
    )
    return instance

synthetic = generate_synthetic(SYNTHETIC)
small = CVRPInstance(
    name="Handwritten teaching example",
    demands=EXAMPLE_DEMANDS.copy(),
    capacity=EXAMPLE_CAPACITY,
    max_vehicles=EXAMPLE_MAX_VEHICLES,
    distances=euclidean_cost_matrix(EXAMPLE_COORDS),
    coordinates=EXAMPLE_COORDS,
    original_ids=np.arange(len(EXAMPLE_DEMANDS)),
)
for inst in (synthetic, small):
    errors = inst.check_input()
    print(f"{inst.name}: n={inst.n_customers}, K={inst.max_vehicles}, "
          f"Q={inst.capacity}, demand={inst.demands[1:].sum():.0f}, "
          f"input_errors={errors}")

### 1.3 · Visual question: where are the customers, and how large are their demands?

The marker sizes represent demand, **not route assignments**. The depot marker is separate. This plot demonstrates why geographic location and payload both matter; spatially close customers might collectively exceed a truck's capacity.

In [ ]:
def plot_customer_map(instance: CVRPInstance, title: str | None = None, label: bool = False):
    """Show spatial inputs; optional coordinates are mandatory for this plot."""
    xy = instance.coordinates
    if xy is None:
        print("No coordinates provided; mapping skipped (distance matrix still usable).")
        return
    fig, ax = plt.subplots(figsize=(9, 6))
    q = instance.demands[1:]
    sizes = 35 + 15 * q / max(1, float(q.max())) * 7
    scatter = ax.scatter(xy[1:, 0], xy[1:, 1], s=sizes,
                         c=q, cmap="viridis", alpha=0.8, edgecolor="black", lw=0.4)
    ax.scatter([xy[0,0]], [xy[0,1]], marker="s", s=220, color="crimson",
               edgecolor="black", label="Depot (0)", zorder=5)
    if label and instance.n_customers <= 55:
        for i, (x, y) in enumerate(xy[1:], start=1):
            ax.annotate(str(i), (x, y), xytext=(3, 3), textcoords="offset points", fontsize=7)
    fig.colorbar(scatter, ax=ax, label="Demand")
    ax.set(xlabel="x coordinate", ylabel="y coordinate",
           title=title or f"Customer locations · {instance.name}")
    ax.set_aspect("equal", adjustable="datalim")
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

plot_customer_map(synthetic)

### 1.4 · Demand profiles and elementary feasibility tests

Two easy necessary conditions for feasibility are:

$$\max_{i\in C}q_i\le Q,\qquad \sum_{i\in C}q_i\le KQ.$$

The capacity lower bound on **vehicles used** is:

$$L_K=\left\lceil\frac{\sum_{i\in C}q_i}{Q}\right\rceil.$$

**Caution:** these checks cannot prove full feasibility because unsplittable demands create a bin-packing restriction.

In [ ]:
q = synthetic.demands[1:]
Q = synthetic.capacity
K = synthetic.max_vehicles
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.bar(np.arange(1, len(q)+1), q, color="steelblue")
ax1.axhline(Q, color="crimson", linestyle="--", label=f"Capacity Q={Q:g}")
ax1.set(xlabel="Customer index", ylabel="Demand", title="Individual customer demands")
ax1.legend()
ax2.hist(q, bins=np.arange(q.min()-0.5, q.max()+1.5, 1), color="darkcyan",
         edgecolor="white")
ax2.set(xlabel="Demand", ylabel="Number of customers", title="Demand frequency distribution")
plt.tight_layout(); plt.show()

lower_bound = math.ceil(float(q.sum()) / Q)
print(f"Total demand: {q.sum():g} · available capacity KQ: {K*Q:g}")
print(f"Necessary minimum trucks from aggregate capacity: {lower_bound}")
print("Individual demands within capacity:", bool((q <= Q).all()))
print("Aggregate demand within total fleet capacity:", bool(q.sum() <= K*Q))

### 1.5 · Why capacities and distribution are separate issues

A vehicle can have spare capacity while all customers remain individually unassignable under the remaining fleet allocation. This is why aggregate capacity alone does not prove feasibility.

Below, **six-unit customers with ten-unit trucks** illustrate how the aggregate capacity bound can be satisfied and the actual packing still impossible. This is a conceptual counterexample, **not** a solution attempt.

In [ ]:
# These three figures are a logical counterexample, not a solver result.
packing_demands = np.array([6, 6, 6])
packing_Q, packing_K = 10, 2
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(["Customer A", "Customer B", "Customer C"], packing_demands,
       color=["#5787b7", "#689b84", "#be8b65"])
ax.axhline(packing_Q, color="crimson", linestyle="--", label="Vehicle capacity")
ax.set(ylabel="Individual demand", title="Necessary capacity condition is not sufficient")
ax.legend()
plt.tight_layout(); plt.show()
print("Sum of demand =", packing_demands.sum(), "≤ KQ =", packing_Q*packing_K)
print("But every pair has demand 12 > Q=10, so 3 vehicles would be required.")

## 2 · Decision variables: what exactly can be chosen?

For every truck $k\in F$ and arc $(i,j)\in A$:

$$x_{ijk}=\begin{cases}1,&\text{if truck }k\text{ travels directly from }i\text{ to }j,\\0,&\text{otherwise.}\end{cases}$$

For every customer $i\in C$ and truck $k\in F$:

$$y_{ik}=\begin{cases}1,&\text{if truck }k\text{ serves customer }i,\\0,&\text{otherwise.}\end{cases}$$

For each truck $k\in F$:

$$z_k=\begin{cases}1,&\text{if truck }k\text{ is used,}\\0,&\text{otherwise.}\end{cases}$$

For every customer $i\in C$ and truck $k\in F$, introduce an auxiliary **visit-order** variable $u_{ik}$.

$$u_{ik}=0\quad\text{if }y_{ik}=0;\qquad
1\le u_{ik}\le n\quad\text{if }y_{ik}=1.$$

The order variable is not an additional delivery decision: it exists to forbid customer-only cycles. It can be **continuous** because strict $+1$ sequencing constraints make it increase on every selected customer-to-customer arc; integer order variables are also allowed. With continuous $u$, the model is more precisely a **Mixed-Integer Linear Program (MILP)**, also commonly discussed under the broader PLNE/ILP heading.

**Why three kinds of binary variable?** $x$ expresses route *order*, $y$ identifies *assignment/load*, and $z$ distinguishes an *unused* truck from a real depot-to-depot route. These links will be enforced, not assumed.

### 2.1 · Visual interpretation of an arc decision

The diagram below illustrates one **hypothetical** truck route and the corresponding binary choices. It does **not** imply any optimization has happened.

In [ ]:
# Conceptual illustration drawn from the hand-written route [1,2,3].
path = [0, 1, 2, 3, 0]
xy = small.coordinates
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(xy[1:,0], xy[1:,1], s=110, color="lightsteelblue", edgecolor="black", zorder=3)
ax.scatter(xy[0,0], xy[0,1], s=230, marker="s", color="crimson", zorder=5)
for i,(x,y) in enumerate(xy):
    ax.text(x+2, y+1, f"{i}" + (" (depot)" if i==0 else ""), fontsize=9)
for a,b in zip(path[:-1], path[1:]):
    ax.annotate("", xy=xy[b], xytext=xy[a],
                arrowprops=dict(arrowstyle="->", color="darkblue", lw=2, shrinkA=7, shrinkB=7))
ax.set(xlabel="x", ylabel="y", title="If x[0,1,k]=x[1,2,k]=x[2,3,k]=x[3,0,k]=1 ...")
ax.set_aspect("equal", adjustable="datalim")
plt.tight_layout(); plt.show()
print("Active arcs in this illustrative route:", list(zip(path[:-1], path[1:])))

## 3 · Objective: minimize the **total distance traveled by all trucks**

$$\boxed{\min_{x,y,z,u}\; Z=\sum_{k\in F}\sum_{(i,j)\in A}d_{ij}x_{ijk}}$$

Each selected arc incurs its cost *once*. We include depot-to-customer, customer-to-customer, and customer-to-depot travel. The objective sums over all trucks; it does **not** minimize the longest route, the number of trucks, or delivery time.

Even if coordinates are available, the matrix $D=[d_{ij}]$ is the authoritative cost input. A road-based matrix may be asymmetric ($d_{ij}\neq d_{ji}$); the formulation still works. Standard symmetric Euclidean CVRPLIB examples are only a special case.

In [ ]:
# Visualize the distance matrix. A small subset makes labels readable, while
# the full matrix remains available for mathematical evaluation and validation.
D = synthetic.distances
shown = min(D.shape[0], 19)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(D[:shown, :shown], cmap="magma", interpolation="nearest")
fig.colorbar(im, ax=ax, label="Distance / cost")
ax.set(xlabel="Destination node j", ylabel="Origin node i",
       title=f"Pairwise costs · first {shown} nodes")
ax.set_xticks(np.arange(shown)); ax.set_yticks(np.arange(shown))
plt.tight_layout(); plt.show()

## 4 · CVRPTW routing and capacity constraints (M0–M10; timing in §4.5)

Let $C=\{1,\dots,n\}$, $V=C\cup\{0\}$, $F=\{1,\dots,K\}$, and $A=\{(i,j)\in V^2:i\neq j\}$.

The confirmed formulation uses **at most K** vehicles. No constraint forces all trucks to operate. All share Q and the depot operating window.

### (M0) Objective

$$\min\sum_{k\in F}\sum_{(i,j)\in A}d_{ij}x_{ijk}.$$

### (M1) Each customer is assigned exactly once

$$\boxed{\sum_{k\in F} y_{ik}=1\quad\forall i\in C.}$$

### (M2–M3) Each assigned truck enters and leaves its customers exactly once

$$\boxed{\sum_{j\in V\setminus\{i\}}x_{ijk}=y_{ik}
\quad\forall i\in C,\;\forall k\in F,}$$

$$\boxed{\sum_{j\in V\setminus\{i\}}x_{jik}=y_{ik}
\quad\forall i\in C,\;\forall k\in F.}$$

Together with (M1), these encode visit-once and conservation of flow.

### (M4–M5) Every used truck departs from and returns to the depot

$$\boxed{\sum_{j\in C}x_{0jk}=z_k\quad\forall k\in F,}$$

$$\boxed{\sum_{i\in C}x_{i0k}=z_k\quad\forall k\in F.}$$

A truck with $z_k=0$ has no assigned customers or selected arcs.

### (M6) Capacity of each truck

$$\boxed{\sum_{i\in C}q_i y_{ik}\le Q z_k\quad\forall k\in F.}$$

Since each customer has one assigned truck, no demand is split across trucks.

### (M7–M9) Order-based Miller–Tucker–Zemlin (MTZ) subtour elimination

$$\boxed{y_{ik}\le u_{ik}\le n y_{ik}
\quad\forall i\in C,\;\forall k\in F,}$$

$$\boxed{u_{jk}\ge u_{ik}+1-(n+1)(1-x_{ijk})
\quad\forall i,j\in C,\;i\ne j,\;\forall k\in F.}$$

When $x_{ijk}=1$, order must increase by at least 1. A cycle of customers would require $u_{i_1,k}<u_{i_2,k}<\cdots<u_{i_m,k}<u_{i_1,k}$, which is impossible. When $x_{ijk}=0$, the lower bound is redundant because $u_{ik}\le n$ and $u_{jk}\ge0$; the **valid big-$M$ coefficient is $n+1$**.

**Strength:** this order formulation still eliminates disconnected cycles if some demands $q_i=0$, unlike a naïve positive-demand load-based MTZ proof.

### (M10) Decision domains / forbidden self-loops

$$\boxed{x_{ijk}\in\{0,1\}\quad\forall (i,j)\in A,\ k\in F,}$$
$$\boxed{y_{ik}\in\{0,1\}\quad\forall i\in C,\ k\in F,}$$
$$\boxed{z_k\in\{0,1\}\quad\forall k\in F,}$$
$$\boxed{u_{ik}\in\mathbb R_{\ge0}\quad\forall i\in C,\ k\in F.}$$

There are **no** $x_{iik}$ variables (or equivalently, set them to 0). We assume all input distances are finite and all truck capacities identical.

### Optional (not required): symmetry reduction

$$z_k\ge z_{k+1}\quad (k=1,\dots,K-1).$$

Trucks are interchangeable, so unused trucks can be placed after active trucks. This changes *which truck label* is used, not the objective or set of physical routes. It is an optional improvement to a future ILP implementation, not a needed feasibility rule.

### 4.1 · Constraint-to-business-rule traceability

| Mathematical rule | Operational meaning | Failure if omitted |
|---|---|---|
| M1: $\sum_k y_{ik}=1$ | Every customer served once | Skipped or multiply assigned customers |
| M2–M3: incoming/outgoing $=y_{ik}$ | Continuity and visit-once per truck | Fragmented or unmatched travel arcs |
| M4–M5: depot degree $=z_k$ | Active route begins and ends at depot | Open routes / unused truck inconsistencies |
| M6: $\sum_i q_i y_{ik}\le Qz_k$ | Payload capacity | Overloaded trucks |
| M7–M9: $u$ bounds and increasing order | No customer-only cycles | Disconnected subtours |
| M10: binary domains and $i\ne j$ | Yes/no travel decisions and no self-loops | Fractional or meaningless trips |

These equations establish routing and capacity. Section 4.5 adds service duration, hard service-start windows, and depot schedules. No fixed route-length constraint is assumed.

### 4.2 · Visual proof: why flow constraints alone do not eliminate subtours

Imagine one truck with a valid depot loop $0\to1\to2\to0$ and a disconnected customer loop $3\to4\to3$. Every customer still has one incoming and one outgoing arc, but the second loop is unreachable from the depot. It is **not** a valid delivery plan.

In [ ]:
# Purely schematic node positions; edges deliberately exhibit a bad subtour.
positions = {0:(0,0), 1:(-1,1), 2:(-1,-1), 3:(3,1), 4:(4,0)}
bad_arcs = [(0,1),(1,2),(2,0),(3,4),(4,3)]
fig, ax = plt.subplots(figsize=(9, 4.5))
for v,(x,y) in positions.items():
    ax.scatter([x],[y],s=380,marker="s" if v==0 else "o",
               color="crimson" if v==0 else "steelblue",zorder=5)
    ax.text(x,y,str(v),color="white",ha="center",va="center",fontsize=11,zorder=6)
for a,b in bad_arcs:
    ax.annotate("", xy=positions[b], xytext=positions[a],
                arrowprops=dict(arrowstyle="->", lw=2,
                                color="firebrick" if a>=3 else "black",
                                connectionstyle="arc3,rad=0.19", shrinkA=19, shrinkB=19))
ax.text(2.5,-1.25,"Disconnected customer cycle → INVALID",color="firebrick",weight="bold")
ax.set(title="Degree / flow conservation alone is insufficient", xlim=(-2,5.4),ylim=(-1.8,2))
ax.axis("off")
plt.tight_layout(); plt.show()

### 4.3 · Why MTZ eliminates the disconnected cycle

If the bad tour uses arcs $3\to4$ and $4\to3$ for truck $k$, the order equations imply

$$u_{4k}\ge u_{3k}+1\quad\text{and}\quad
u_{3k}\ge u_{4k}+1.$$

Adding gives $0\ge2$, a contradiction. This argument applies to any customer-only cycle, even with zero-demand customers.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.scatter([1,3], [1,1], s=1200, c=["steelblue","darkorange"], edgecolor="black")
ax.text(1,1,"u₃",color="white",ha="center",va="center",fontsize=16)
ax.text(3,1,"u₄",color="white",ha="center",va="center",fontsize=16)
ax.annotate("u₄ ≥ u₃ + 1",(2.8,1.25),(1.25,1.25),
            arrowprops={"arrowstyle":"->","lw":2},ha="center",va="bottom")
ax.annotate("u₃ ≥ u₄ + 1",(1.2,.70),(2.8,.70),
            arrowprops={"arrowstyle":"->","lw":2},ha="center",va="top")
ax.text(2,-.05,"Together: 0 ≥ 2 → contradiction",ha="center",color="crimson",fontsize=13)
ax.set(xlim=(0,4),ylim=(-.5,1.9),title="MTZ imposes an impossible increasing order on a directed cycle")
ax.axis("off")
plt.tight_layout(); plt.show()

### 4.4 · Why this formulation is sufficient and necessary

**(Necessary)** Take any physically feasible CVRPTW solution. Assign $z_k=1$ exactly for used trucks; set $y_{ik}=1$ exactly on the truck that serves customer $i$; set $x_{ijk}=1$ on consecutive route arcs, including the depot legs; set $u_{ik}$ to the position (1,2,...) of $i$ on truck $k$, or 0 when unassigned. All the equations hold.

**(Sufficient)** M1 assigns every customer to exactly one truck; M2/M3 force exactly one incoming and outgoing arc for the assigned customer; M4/M5 give each used truck exactly one depot departure and return; M7–M9 forbid every customer-only subtour. Hence the selected arcs for each used truck form exactly one connected directed route from and back to the depot. M6 enforces capacity. Therefore every feasible model assignment corresponds to a route satisfying the routing and capacity constraints.

**Timing requirement:** Section 4.5 must also hold for a complete CVRPTW solution.

**Representation limitation:** these statements assume all valid required arcs can be represented in $A$ and each distance is well-defined. In this notebook the full directed graph is used. The model does not imply symmetry or a triangle inequality unless the data supply them.

### 4.5 · Complete static CVRPTW MILP: temporal extension

Retain **all** M0–M10 above, including continuous order variables u, binary x/y/z, forbidden self-loops and the distance objective. Add y_ik≤z_k explicitly. This is MILP (mixed-integer PLNE terminology), not an all-integer model. Costs d_ij and travel times t_ij are separate static parameters; for the selected CVRPLIB cases t=d, each Euclidean arc truncated to one decimal. Coordinate symbols are p_i=(X_i,Y_i), distinct from window bounds.

| Data | Definition |
|---|---|
| [a_i,b_i], s_i | Hard service-start interval and nonnegative service duration, each customer |
| [a_0,b_0], H=b_0 | Depot opening, latest return and finite time horizon |
| d_ij,t_ij | Finite nonnegative static distance and travel time for each allowed arc |
| Input domains | 0≤a_i≤b_i, 0≤a_0≤H, s_i≥0, depot s_0=0; one depot |

| New variables | Domain and meaning |
|---|---|
| A_ik, S_ik, D_ik | Arrival, service start, departure from customer i by vehicle k |
| W_ik=S_ik−A_ik | Derived waiting, nonnegative |
| T_k,R_k | Separate departure from and return to depot; do not reuse one depot time variable |

All times use **benchmark elapsed-time units**, not assumed minutes or time zones. Zero all schedule variables for inactive assignments/vehicles. For all i∈C,k∈F:

$$0\le A_{ik},S_{ik},D_{ik}\le H y_{ik},\quad y_{ik}\le z_k.\tag{TW1}$$
$$a_i y_{ik}\le S_{ik}\le b_i y_{ik},\quad S_{ik}\ge A_{ik},\quad D_{ik}=S_{ik}+s_i y_{ik}.\tag{TW2}$$
$$a_0z_k\le T_k\le R_k\le H z_k.\tag{TW3}$$

For any customer-to-customer arc i→j, enforce exact physical arrival by two-sided activation:
$$-M_{ij}(1-x_{ijk})\le A_{jk}-D_{ik}-t_{ij}\le M_{ij}(1-x_{ijk}).\tag{TW4}$$
Use distinct depot legs:
$$-M_{0j}(1-x_{0jk})\le A_{jk}-T_k-t_{0j}\le M_{0j}(1-x_{0jk}),\tag{TW5}$$
$$-M_{i0}(1-x_{i0k})\le R_k-D_{ik}-t_{i0}\le M_{i0}(1-x_{i0k}).\tag{TW6}$$

**Valid finite bounds:** choose M_ij=H+t_ij. Each difference is between −H−t_ij and H−t_ij because all its time variables lie in [0,H]; hence both inequalities deactivate when x=0. When x=1 equality enforces sufficient travel exactly, with any waiting represented at the destination. No arbitrary huge constant is needed. MTZ still uses its separately derived n+1 coefficient. Omit unreachable arcs rather than putting infinity in M. Retain MTZ even for zero travel/service or coincident coordinates; temporal ordering alone need not exclude zero-time cycles.

**Window interpretation:** arrival 30, opening 40, closing 50, service 15 → wait 10, start 40, depart 55 is allowed if the remaining route returns before depot close. Arrival 51 is infeasible. Completion after b_i is permitted; no service **starts** outside [a_i,b_i]. If a_i>H or service cannot finish before return, the relevant assignment is infeasible; do not clip raw windows to make it pass.

**Correctness:** any valid scheduled depot tour induces M0–M10 and TW1–TW6, with u equal to visit position. Conversely, routing constraints/MTZ give one connected tour per active truck; TW4–TW6 enforce each actual travel leg, TW2 enforces service and nonnegative waiting, TW3 enforces depot opening/return and zero inactive times. A single depot time would incorrectly constrain a positive-duration cycle, which is why T and R differ.

The MILP allows extra pre-service waiting. Because travel is static and distance is the objective, every feasible route sequence also admits its earliest schedule: set T=a_0 and S=max(A,a_i) recursively. This cannot make service or return later, so checking that schedule suffices to establish temporal feasibility of a supplied order. It does not construct or optimize a route. Inactive trucks have no schedule, no arcs and zero cost.

**Domains and scope:** A,S,D,T,R are continuous nonnegative variables; W=S−A is derived, u is continuous with M7–M9 bounds, x/y/z are binary, and no self-loops exist. TW4 applies only to distinct customer pairs; TW5/TW6 cover depot legs. All input times are nonnegative finite elapsed units. Each vehicle may choose T within the common depot window; default checker departure a_0 supplies a feasible representative, not an optimization of departure. Later feasible departures are explicitly supported and tested. Symmetry reduction is optional because vehicles share every modeled constraint. The geometry/capacity examples above isolate the routing foundation; the synthetic examples below check the full temporal extension.


## 5 · Complexity: static CVRPTW is NP-hard

Start with finite nonnegative rational TSP distances on n+1 nodes and a distance budget B. Keep those distances and use t=d, one truck, unit customer demands, Q=n, zero service and all windows [0,H], with H=(n+1) max(d_ij) (H=0 is allowed for the all-zero degenerate case). Any simple depot-returning tour has at most n+1 arcs and fits this horizon. The construction has polynomial encoding size. A CVRPTW route of distance at most B exists exactly when the original TSP tour does. Thus optimization is NP-hard. A supplied route list has a polynomial-time exact rational earliest-service/capacity/cost verification, so the finite-rational distance-budget decision variant is NP-complete.

The formulation has O(K n²) routing/MTZ and temporal constraints and O(Kn) schedule variables. This counts model size, not execution time. Heuristic or metaheuristic research belongs to a later phase; none is chosen here. The data top out at 1,000 customers and cannot alone establish several-thousand-customer capability.


## 6 · Independent feasibility validator (not an optimization algorithm)

A **route list** is a set of customer sequences. Example: `[[1, 2], [3, 4]]` represents $0\to1\to2\to0$ and $0\to3\to4\to0$. The depot is inserted by the checker and therefore must **not** appear within any customer sequence.

The following functions do **not search for routes**. They only assess routes provided by a human or benchmark source, identifying customer coverage, duplicated visits, capacity, depot, fleet and distance issues.

In [ ]:
@dataclass
class ValidationResult:
    feasible: bool
    problems: list[str]
    route_loads: list[float]
    route_distances: list[float]
    total_distance: float
    customers_seen: int


def validate_routes(instance: CVRPInstance, routes: list[list[int]]) -> ValidationResult:
    """Validate an EXISTING route list; it never modifies or optimizes it.

    Every route is a customer-only sequence. It must contain valid integer IDs
    from 1...n; the depot (0) is implicit. Empty routes are ignored, so an
    available but unused truck is permitted. A vehicle is assigned at most one
    nonempty sequence, matching the fleet model.
    """
    if hasattr(instance, 'time_windows'):
        raise TypeError("Use check_cvrptw for CVRPTWInstance; this checker is capacity-only teaching code")
    issues = instance.check_input()
    if issues:
        return ValidationResult(False, issues, [], [], float("nan"), 0)
    loads, route_distances, visited = [], [], []
    active_routes = [r for r in routes if len(r) > 0]
    if len(active_routes) > instance.max_vehicles:
        issues.append(f"Fleet limit: used {len(active_routes)} > K={instance.max_vehicles}.")
    for vehicle_idx, route in enumerate(active_routes, start=1):
        invalid = [node for node in route if type(node) not in (int, np.int32, np.int64)
                   or not (1 <= int(node) <= instance.n_customers)]
        if invalid:
            issues.append(f"Route {vehicle_idx}: invalid customer/depot IDs: {invalid}.")
            loads.append(float("nan")); route_distances.append(float("nan"))
            continue
        visited.extend(route)
        load = float(np.sum(instance.demands[route]))
        loads.append(load)
        if load > instance.capacity + 1e-9:
            issues.append(f"Route {vehicle_idx}: load {load:g} exceeds Q={instance.capacity:g}.")
        path = [0] + list(route) + [0]
        route_distance = sum(float(instance.distances[a, b])
                             for a, b in zip(path[:-1], path[1:]))
        route_distances.append(route_distance)
    counts = Counter(visited)
    duplicate = sorted(i for i, number in counts.items() if number > 1)
    missing = sorted(set(range(1, instance.n_customers + 1)) - set(visited))
    if duplicate:
        issues.append(f"Duplicate customer(s): {duplicate}.")
    if missing:
        issues.append(f"Missing customer(s): {missing}.")
    total = float(sum(route_distances))
    return ValidationResult(
        feasible=not issues, problems=issues, route_loads=loads,
        route_distances=route_distances, total_distance=total,
        customers_seen=len(visited),
    )


def summarize_route_check(instance: CVRPInstance, routes: list[list[int]]) -> pd.DataFrame:
    result = validate_routes(instance, routes)
    print("Feasible:", result.feasible)
    print("Distance:", round(result.total_distance, 3))
    print("Issues:", result.problems if result.problems else "none")
    table = pd.DataFrame({
        "Truck": range(1, len(result.route_loads) + 1),
        "Load": result.route_loads,
        "Capacity": instance.capacity,
        "Distance": result.route_distances,
    })
    display(table)
    return table

small_check_table = summarize_route_check(small, EXAMPLE_ROUTES)
assert validate_routes(small, EXAMPLE_ROUTES).feasible

### 6.1 · Plot two explicitly supplied hand-written routes

The plotted routes are **not the solution of an optimizer**. They demonstrate exactly how the mathematical model interprets a valid route list and its distance cost.

In [ ]:
def plot_routes(instance: CVRPInstance, routes: list[list[int]], title: str):
    """Draw only user-supplied or reference routes. No route construction."""
    if instance.coordinates is None:
        print("Skipping route drawing: this instance does not include coordinates.")
        return
    fig, ax = plt.subplots(figsize=(9, 6))
    xy = instance.coordinates
    colors = plt.get_cmap("tab10")
    ax.scatter(xy[1:,0], xy[1:,1], s=55, color="white",
               edgecolor="black", zorder=5)
    ax.scatter(xy[0,0], xy[0,1], s=250, marker="s",
               color="crimson", edgecolor="black", zorder=6, label="Depot")
    for i,(x,y) in enumerate(xy[1:],start=1):
        if instance.n_customers <= 45:
            ax.text(x+1.2, y+1.2, str(i), fontsize=8)
    for k, route in enumerate(routes):
        if not route:
            continue
        path = [0] + list(route) + [0]
        color = colors(k % 10)
        for a,b in zip(path[:-1],path[1:]):
            ax.plot(xy[[a,b],0],xy[[a,b],1], color=color,alpha=.85,lw=2)
        ax.plot([],[],color=color,lw=2,label=f"Truck {k+1}")
    ax.set(xlabel="x",ylabel="y",title=title)
    ax.set_aspect("equal",adjustable="datalim")
    ax.legend(loc="upper left",bbox_to_anchor=(1.02,1))
    plt.tight_layout(); plt.show()

plot_routes(small, EXAMPLE_ROUTES, "Hand-written valid routes (not optimized)")

### 6.2 · Load profiles and capacity slack

Because the rules constrain **per-route demand**, not aggregate demand alone, it is useful to visualize individual vehicle loads and unused capacity. All loads below are calculated from the hand-written route list.

In [ ]:
result_small = validate_routes(small, EXAMPLE_ROUTES)
loads = np.array(result_small.route_loads)
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(loads))
ax.bar(x, loads, label="Delivered demand", color="steelblue")
ax.bar(x, small.capacity - loads, bottom=loads, label="Unused capacity", color="lightgray")
ax.axhline(small.capacity, color="crimson", linestyle="--", label="Capacity Q")
ax.set_xticks(x, [f"Truck {i+1}" for i in x])
ax.set(ylabel="Demand", title="Load composition of manually supplied routes")
ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

### 6.3 · Test **invalid** user-supplied routes

These tests are deliberate counterexamples: omitted customers, a duplicate delivery, fleet overuse, and a load exceeding $Q$. They verify that the checker rejects each violation and explain the business consequence.

In [ ]:
# Each case is explicitly constructed to violate a particular rule.
# Notice: test cases change the INPUT ROUTES, not the solver (there is none).
cases = {
    "valid manual routes": [[1,2,3], [4,5,6]],
    "one customer missing": [[1,2,3], [4,5]],
    "duplicate customer": [[1,2,3], [4,5,6,1]],
    "over-capacity route": [[1,2,3,4], [5,6]],
    "depot used as customer": [[0,1,2,3], [4,5,6]],
    "too many trucks": [[1],[2],[3],[4,5,6]],
}
case_rows = []
for name, routes in cases.items():
    result = validate_routes(small, routes)
    case_rows.append({"Scenario":name,"Feasible":result.feasible,
                      "Errors":len(result.problems),
                      "Reason":"; ".join(result.problems) if result.problems else "—"})
scenario_table = pd.DataFrame(case_rows)
display(scenario_table)
assert scenario_table["Feasible"].tolist() == [True,False,False,False,False,False]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(scenario_table["Scenario"], scenario_table["Errors"],
        color=["forestgreen" if good else "firebrick" for good in scenario_table["Feasible"]])
ax.set(xlabel="Detected problems", title="Constraint violation detection: supplied scenarios")
ax.invert_yaxis()
plt.tight_layout(); plt.show()

### 6.4 · Reconstruct model variables from supplied routes, then verify the equations

A stronger check does not merely examine routes. It builds explicit $x$, $y$, $z$, and $u$ arrays from a **given** route list, and directly evaluates the ILP equations, including every MTZ inequality. This is useful for catching errors in the proposed mathematical notation **without running an optimizer**.

In [ ]:
def certify_model_equations(instance: CVRPInstance, routes: list[list[int]]) -> dict:
    """Check every mathematical condition on a PROVIDED valid-route candidate.

    This is an exhaustive evaluator for equations M1-M10 on the input
    assignment; it is NOT a general ILP solver (no search, no branching,
    no route optimization). O(K*n^2) time is acceptable for small examples.
    """
    n, K = instance.n_customers, instance.max_vehicles
    V = n + 1
    if len([r for r in routes if r]) > K:
        raise ValueError("Cannot encode more routes than available trucks")
    x = np.zeros((K, V, V), dtype=int)
    y = np.zeros((K, V), dtype=int)
    z = np.zeros(K, dtype=int)
    u = np.zeros((K, V), dtype=float)
    active = [r for r in routes if r]
    for k, route in enumerate(active):
        z[k] = 1
        for position, customer in enumerate(route, start=1):
            if customer < 1 or customer > n:
                raise ValueError("Invalid customer ID")
            y[k, customer] = 1
            u[k, customer] = float(position)
        full_path = [0] + route + [0]
        for i, j in zip(full_path[:-1], full_path[1:]):
            x[k, i, j] = 1

    check = {}
    check["M1: each customer once"] = bool(np.all(y[:,1:].sum(axis=0) == 1))
    check["M2: outbound = assignment"] = bool(np.all(x[:,:, :].sum(axis=2)[:,1:] == y[:,1:]))
    check["M3: inbound = assignment"] = bool(np.all(x.sum(axis=1)[:,1:] == y[:,1:]))
    check["M4: leave depot"] = bool(np.all(x[:,0,1:].sum(axis=1) == z))
    check["M5: return to depot"] = bool(np.all(x[:,1:,0].sum(axis=1) == z))
    check["M6: capacities"] = bool(np.all(y[:,1:] @ instance.demands[1:] <= instance.capacity*z + 1e-9))
    check["M7: u lower bound"] = bool(np.all(y[:,1:] <= u[:,1:] + 1e-9))
    check["M8: u upper bound"] = bool(np.all(u[:,1:] <= n*y[:,1:] + 1e-9))
    mtz_ok = True
    for k in range(K):
        for i in range(1,V):
            for j in range(1,V):
                if i != j:
                    lhs = u[k,j]
                    rhs = u[k,i] + 1 - (n+1)*(1-x[k,i,j])
                    if lhs < rhs - 1e-9:
                        mtz_ok = False
                        break
    check["M9: subtour (MTZ)"] = mtz_ok
    check["M10: no self-loop"] = bool(np.all(np.diagonal(x,axis1=1,axis2=2)==0))
    cost = float(np.sum(x * instance.distances[None,:,:]))
    if hasattr(instance, 'time_windows'):
        from scripts.cvrptw_model import check_cvrptw
        route_cost = check_cvrptw(instance, routes)['distance']
    else:
        route_cost = validate_routes(instance, routes).total_distance
    check["Objective equals route cost"] = bool(np.isclose(cost,route_cost))
    return {"checks":check, "all_hold":all(check.values()), "cost":cost,
            "x":x,"y":y,"z":z,"u":u}

certificate = certify_model_equations(small, EXAMPLE_ROUTES)
display(pd.DataFrame({"Equation": list(certificate["checks"]),
                      "Holds": list(certificate["checks"].values())}))
assert certificate["all_hold"]
print("Verified ILP objective:", round(certificate["cost"],3))

### 6.5 · What happens when demand is zero?

Zero-demand customers are legal inputs in the mathematical model. Subtour elimination **must still work**. Our visit-order-based MTZ uses $+1$ rather than $+q_i$ and is therefore not dependent on positive demands.

In [ ]:
# Small zero-demand edge case: each customer still requires a visit.
zero_case = CVRPInstance(
    name="Zero-demand edge case",
    demands=np.array([0., 0., 0., 3.]), capacity=3., max_vehicles=2,
    distances=euclidean_cost_matrix(np.array([[0.,0.],[1.,0.],[2.,1.],[2.,2.]])),
    coordinates=np.array([[0.,0.],[1.,0.],[2.,1.],[2.,2.]]),
)
zero_routes = [[1,2,3]]   # Illustrative handwritten candidate.
print("Zero-demand case feasible:", validate_routes(zero_case,zero_routes).feasible)
print("MTZ and all model equations hold:", certify_model_equations(zero_case,zero_routes)["all_hold"])
assert certify_model_equations(zero_case,zero_routes)["all_hold"]

## 8 · Parameter sensitivity without solving the CVRPTW

The following studies vary capacity, fleet size, customer count, and spatial distribution to illustrate necessary feasibility conditions and model size. They do not generate optimized routes.

### 8.1 · Capacity $Q$ versus the aggregate-demand fleet bound

For total demand $D$, $\lceil D/Q\rceil$ is a necessary lower bound on vehicles. Time windows can require more vehicles.

In [ ]:
D_total = float(synthetic.demands[1:].sum())
max_q = float(synthetic.demands[1:].max())
# Study capacities from at least the largest individual demand up to an editable
# multiple of the configured capacity. These are experiments, not model constants.
cap_grid=np.linspace(max(1,max_q),max(max_q+1,2*SYNTHETIC.vehicle_capacity),60)
truck_lb=np.ceil(D_total/cap_grid)
fig,ax=plt.subplots(figsize=(9,4.5))
ax.step(cap_grid,truck_lb,where="post",lw=2,label="Aggregate-demand lower bound")
ax.axhline(SYNTHETIC.max_vehicles,color="crimson",linestyle="--",label="Configured fleet K")
ax.axvline(SYNTHETIC.vehicle_capacity,color="gray",linestyle=":",label="Configured capacity Q")
ax.set(xlabel="Candidate vehicle capacity Q",ylabel="Necessary minimum vehicles",
       title="Capacity sensitivity: a bound, not a solution")
ax.legend()
plt.tight_layout();plt.show()

### 8.2 · How does the full ILP formulation scale?

For a complete directed graph with $n$ customers and $K$ trucks, our formulation creates:

- $K(n+1)n$ binary arc variables $x$ (no diagonal),
- $Kn$ binary assignment variables $y$,
- $K$ binary vehicle-use variables $z$,
- $Kn$ continuous order variables $u$,
- $3Kn$ continuous customer times $A,S,D$ and $2K$ depot times $T,R$; waiting $W=S-A$ is derived.

Therefore the number of binary variables is $K(n^2+2n+1)=K(n+1)^2$, and continuous variables number $4Kn+2K$. The number of order inequalities alone is $Kn(n-1)$ — quadratic in customers and linear in vehicles. This is why we **model with ILP now**, but should later use an efficient heuristic for large instances.

These are **representation sizes**, not execution time predictions and not experimental algorithm results.

In [ ]:
def model_sizes(n: int, k: int) -> dict[str,int]:
    """Count decision variables and major constraint families algebraically."""
    if n<1 or k<1:
        raise ValueError("n and K must be positive")
    return {
        "arcs (binary)":k*(n+1)*n,
        "assignments (binary)":k*n,
        "vehicle-use (binary)":k,
        "order (continuous)":k*n,
        "customer times (continuous)":3*k*n,
        "depot times (continuous)":2*k,
        "temporal arc inequalities":2*k*n*(n+1),
        "MTZ pairwise inequalities":k*n*(n-1),
        "degree equalities":2*k*n,
        "customer assignment eqs":n,
        "depot equalities":2*k,
        "capacity inequalities":k,
        "order-bound inequalities":2*k*n,
    }

sizes=model_sizes(synthetic.n_customers,synthetic.max_vehicles)
display(pd.DataFrame(sizes.items(),columns=["Component","Count"]))
assert sizes["arcs (binary)"] == synthetic.max_vehicles*(synthetic.n_customers+1)*synthetic.n_customers
assert sum(sizes[k] for k in ["arcs (binary)","assignments (binary)","vehicle-use (binary)"]) == synthetic.max_vehicles*(synthetic.n_customers+1)**2
assert sum(v for key,v in sizes.items() if "(continuous)" in key) == 4*synthetic.max_vehicles*synthetic.n_customers+2*synthetic.max_vehicles


In [ ]:
# Compare symbolic model sizes for three possible fleet-scaling scenarios.
n_grid=np.unique(np.geomspace(10,2000,num=30).astype(int))
fig,ax=plt.subplots(figsize=(9,5))
for assumed_K in (5,15,40):  # Scenario values only; not hard-coded fleet limits.
    arc_vars=[model_sizes(int(n),assumed_K)["arcs (binary)"] for n in n_grid]
    ax.plot(n_grid,arc_vars,marker=".",label=f"Illustrative K={assumed_K}")
ax.set(xlabel="Number of customers n",ylabel="Number of binary arc variables",
       title="Arc variables in the full MILP scale as O(Kn²)",xscale="log",yscale="log")
ax.legend()
plt.tight_layout();plt.show()

### 8.3 · Spatial distribution: uniform vs clustered customers

Customer geometry is part of the **input**, not a mathematical constraint: uniform and clustered instances can have the same number of customers, identical capacities and similar demand distributions, yet very different routing structure.

In [ ]:
# Use the same configurable values and random seed, changing only layout.
from dataclasses import replace
uniform_demo = generate_synthetic(replace(SYNTHETIC, layout="uniform"))
clustered_demo = generate_synthetic(replace(SYNTHETIC, layout="clustered"))
fig,axs=plt.subplots(1,2,figsize=(12.5,5))
for ax,inst,label in zip(axs,[uniform_demo,clustered_demo],["Uniform", "Clustered"]):
    xy=inst.coordinates
    ax.scatter(xy[1:,0],xy[1:,1],s=40+10*inst.demands[1:],
               c=inst.demands[1:],cmap="viridis",edgecolor="black",lw=.3)
    ax.scatter([xy[0,0]],[xy[0,1]],s=200,marker="s",color="crimson")
    ax.set(xlabel="x",ylabel="y",title=f"{label} inputs · n={inst.n_customers}")
    ax.set_aspect("equal",adjustable="datalim")
plt.tight_layout();plt.show()

### 8.4 · Fleet-and-capacity necessary feasibility map

The heatmap below marks whether $KQ\ge\sum_i q_i$ — an **aggregate necessity test only**. A green cell means *not ruled out* by total capacity; it does **not** prove route feasibility. A red cell is definitely infeasible by this bound.

In [ ]:
qsum=float(synthetic.demands[1:].sum())
max_q=float(synthetic.demands[1:].max())
K_values=np.arange(1,max(9,SYNTHETIC.max_vehicles+4))
Q_values=np.linspace(max(1,max_q),max(2*SYNTHETIC.vehicle_capacity,max_q+1),23)
feasible_capacity=np.array([[k*q>=qsum and q>=max_q for q in Q_values] for k in K_values])
fig,ax=plt.subplots(figsize=(10,5))
img=ax.imshow(feasible_capacity,aspect="auto",origin="lower",cmap="RdYlGn",
              interpolation="nearest",extent=[Q_values[0],Q_values[-1],K_values[0]-.5,K_values[-1]+.5])
ax.scatter([synthetic.capacity],[synthetic.max_vehicles],marker="x",s=180,
           color="black",label="Configured example")
ax.set(xlabel="Vehicle capacity Q",ylabel="Fleet upper bound K",
       title="Green = necessary aggregate capacity condition passes (not sufficient)")
ax.set_yticks(K_values)
ax.legend()
plt.tight_layout();plt.show()

## 9 · Edge cases and model boundaries

| Situation | Expected behavior |
| --- | --- |
| Zero demand | Customer still requires exactly one visit |
| Demand equal to capacity | That vehicle cannot carry another positive demand |
| Early arrival | Vehicle waits until the service window opens |
| Service start after closing | Route is infeasible |
| Late depot return | Route is infeasible |
| Unused vehicle | No assigned customers or arcs |
| Demand above fleet capacity | Instance is infeasible |

Capacity and customer coverage alone cannot establish feasibility: every route also needs a valid schedule.

## 10 · Load selected static CVRPTW files

The configurable root defaults to `data/benchmarks/vrptw/`. `load_cvrptw` below uses the default VRPLIB parser, checks the one-based disk IDs and keeps paired `.sol` labels unchanged. Missing files produce NOT RUN in normal mode and fail strict mode. No benchmark results are synthesized.


## 10A · Complete dataset and schedule validation

Use only the [PyVRP/Instances VRPTW collection](https://github.com/PyVRP/Instances/tree/main/VRPTW): 56 Solomon plus 300 Gehring–Homberger pairs, organized locally as Solomon/GH200/GH400/GH600/GH800/GH1000. Upstream 1,000-customer files are at the VRPTW root; the downloader places them in GH1000. N includes depot. Coordinates `(N,2)`, demands `(N,)`, windows `(N,2)`, service `(N,)`, distance/travel `(N,N)`, scalar K/Q. Scalar SERVICE_TIME expands to customers, with depot zero; a vector must explicitly satisfy depot zero. No assumed GPS, minutes or time zones.

The reusable, commented implementation is in `scripts/cvrptw_model.py`. Optional coordinates/IDs/integer-tenths caches are validated when supplied; route checking works without them. Static travel defaults to distance but is separately configurable. Missing **required** windows/service/data is an explicit error.

The selected disk labels are 1…N, depot 1 → internal 0; supplied `.sol` labels already equal internal customer indices. Never shift twice. DIMACS edge distances are truncated to tenths. Integer-coordinate square-root boundary correction avoids floating rounding at integer tenths. No epsilon promotes a below-boundary distance. Reference cost validation uses integer-tenths sums, not displayed rounded costs. All 356 references are audited when data are available, with hashes; the five plots/examples below are illustrations, not a chosen final test split or optimizer results.


In [ ]:
# Reusable mathematical validation support, not an optimizer.
from scripts.cvrptw_model import (
    CVRPTWInstance, load_cvrptw, check_cvrptw, check_temporal_equations,
    normalize_service, dimacs_tenths, verify_collection, reproducibility_metadata,
)


In [ ]:
metadata = reproducibility_metadata(DATA_ROOT, SYNTHETIC.seed)
display(metadata)
collection = verify_collection(DATA_ROOT, strict=STRICT_DATA)
print("Collection validation:", collection["status"], "checked pairs:", collection["checked"])
reference_audits = {}; records = []
if collection["status"] == "NOT RUN":
    print(collection["reason"])
else:
    assert collection["status"] == "PASS", collection.get("failures")
    print("Collection SHA-256:", collection["collection_sha256"])
    display(collection["results"].groupby("family").agg(pairs=("instance","count"), feasible=("feasible","all")))
    # Supplied illustrations only; no tuning or final benchmark selection.
    for family,name in [("Solomon","C101"),("Solomon","R101"),("Solomon","RC101"),("GH200","C1_2_1"),("GH1000","C1_10_1")]:
        inst=load_cvrptw(DATA_ROOT/family/f"{name}.vrp")
        ref=vrplib.read_solution(str(DATA_ROOT/family/f"{name}.sol"))
        routes=[list(r) for r in ref["routes"]]
        audit=check_cvrptw(inst,routes)
        assert audit["feasible"] and check_temporal_equations(inst,routes,audit)
        reference_audits[name]=(inst,routes,audit)
        records.append(dict(instance=name,customers=inst.n_customers,K=inst.max_vehicles,Q=inst.capacity,
            reference_routes=len(routes),distance=audit["distance"],supplied_cost=ref["cost"],feasible=True))
    display(pd.DataFrame(records))


In [ ]:
if not reference_audits:
    print("Reference plots: NOT RUN (collection unavailable)")
else:
    inst,routes,audit = reference_audits["C101"]
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    xy=inst.coordinates
    sc=axes[0].scatter(xy[1:,0],xy[1:,1],c=inst.time_windows[1:,0],cmap="viridis",s=25)
    axes[0].scatter(*xy[0],marker="s",color="crimson",s=100,label="Depot")
    axes[0].set(title="C101 inputs: customer opening times",xlabel="Benchmark x",ylabel="Benchmark y")
    fig.colorbar(sc,ax=axes[0],label="Opening (benchmark time units)")
    table=audit["schedules"][0]
    for j,row in enumerate(table.itertuples()):
        a,b=inst.time_windows[row.customer]
        axes[1].plot([a,b],[j,j],color="lightgray",lw=7)
        axes[1].scatter(row.arrival,j,marker="x",color="steelblue",zorder=4)
        axes[1].scatter(row.start,j,color="darkgreen",s=18,zorder=3)
    axes[1].plot([],[],"x",color="steelblue",label="Arrival")
    axes[1].plot([],[],"o",color="darkgreen",label="Service start")
    axes[1].set(yticks=range(len(table)),yticklabels=table.customer,title="Published route 1: arrivals and hard start windows",xlabel="Benchmark time units")
    axes[1].legend(); plt.tight_layout(); plt.show()
    display(table)
    display(audit["route_totals"])
    plot_routes(inst,routes,"C101 published reference routes — not found by our code")
    fig,ax=plt.subplots(figsize=(10,4))
    tot=audit["route_totals"]
    for label,color in [("driving","steelblue"),("waiting","darkorange"),("service","seagreen")]:
        bottom=np.zeros(len(tot)) if label=="driving" else tot.driving.to_numpy() if label=="waiting" else (tot.driving+tot.waiting).to_numpy()
        ax.bar(tot.truck,tot[label],bottom=bottom,label=label,color=color)
    assert np.allclose(tot.elapsed,tot.driving+tot.waiting+tot.service)
    ax.set(title="C101 reference schedule duration breakdown",xlabel="Reference truck",ylabel="Benchmark time units")
    ax.legend();plt.tight_layout();plt.show()


In [ ]:
from dataclasses import replace
# Explicit input routes and schedules only: these tests never search for routes.
tiny=CVRPTWInstance(name="Waiting fixture",demands=[0,1],capacity=2,max_vehicles=2,
    distances=[[0,1],[1,0]],time_windows=[[0,10],[5,5]],service_durations=[0,3])
ta=check_cvrptw(tiny,[[1],[]])
assert ta["feasible"] and ta["schedules"][0].iloc[0].waiting==4
assert ta["schedules"][0].iloc[0].departure==8  # Completion after close is allowed.
assert check_temporal_equations(tiny,[[1],[]],ta)
assert check_cvrptw(tiny,[[1]],[4])["feasible"]  # Chosen later depot departure.
assert not check_cvrptw(tiny,[[1]],[5])["feasible"]
assert not check_cvrptw(tiny,[[1]],[-1])["feasible"]
assert not check_cvrptw(replace(tiny,time_windows=[[0,8],[5,5]]),[[1]])["feasible"]
assert check_cvrptw(replace(tiny,time_windows=[[2,9],[5,5]]),[[1]],[2])["feasible"] # Exact depot closing.
for routes in ([], [[1],[1]], [[0,1]], [[2]], [[True]], [[1.0]], [[1],[1],[1]]):
    assert not check_cvrptw(tiny,routes)["feasible"], routes
assert check_cvrptw(replace(tiny,demands=[0,0]),[[1]])["feasible"]
assert check_cvrptw(replace(tiny,capacity=1),[[1]])["feasible"]
zero=replace(tiny,distances=[[0,0],[0,0]],travel_times=[[0,0],[0,0]],service_durations=[0,0],time_windows=[[0,0],[0,0]])
assert check_cvrptw(zero,[[1]])["feasible"]
assert check_temporal_equations(zero,[[1]],check_cvrptw(zero,[[1]]))
assert check_cvrptw(replace(tiny,original_ids=[8,99]),[[1]])["schedules"][0].iloc[0].source_id==99
assert check_cvrptw(replace(tiny,coordinates=None,original_ids=None,distance_tenths=None),[[1]])["feasible"]
overload=CVRPTWInstance(name="Capacity fixture",demands=[0,1,1],capacity=1,max_vehicles=2,
    distances=np.ones((3,3))-np.eye(3),time_windows=[[0,10]]*3,service_durations=[0,0,0])
assert not check_cvrptw(overload,[[1,2]])["feasible"]
assert check_cvrptw(overload,[[1],[2]])["feasible"]
asym=replace(overload,capacity=2,distances=[[0,1,4],[2,0,1],[1,5,0]],travel_times=[[0,1,4],[2,0,1],[1,5,0]])
assert check_cvrptw(asym,[[1,2]])["distance"]==3
assert check_cvrptw(asym,[[2,1]])["feasible"] is False # Return 11 exceeds close 10.
assert np.array_equal(normalize_service(3,3),[0,3,3])
assert np.array_equal(normalize_service([0,2,4],3),[0,2,4])
assert dimacs_tenths([[0,0],[3,4]])[0,1]==50
assert dimacs_tenths([[0,0],[1,1]])[0,1]==14
bad_cases=[dict(demands=1),dict(demands=[0,float('nan')]),dict(max_vehicles=True),dict(max_vehicles=1.5),
    dict(capacity=0),dict(capacity=.5),dict(distances=[[0]]),dict(travel_times=[[0,-1],[1,0]]),
    dict(time_windows=None),dict(time_windows=[[0,10],[6,5]]),dict(service_durations=[1,3]),
    dict(service_durations=[0,-1]),dict(coordinates=[[0,0]]),dict(original_ids=[1,1]),
    dict(distance_tenths=[[0,20],[20,0]])]
for change in bad_cases:
    assert replace(tiny,**change).check_input(),change
    try: check_cvrptw(replace(tiny,**change),[[1]])
    except ValueError: pass
    else: raise AssertionError(change)
# Routing equations and all TW1–TW6 arcs, including unassigned nodes and unused trucks.
fixture=replace(overload,capacity=2,max_vehicles=3)
fa=check_cvrptw(fixture,[[1,2]])
assert certify_model_equations(fixture,[[1,2]])["all_hold"]
assert check_temporal_equations(fixture,[[1,2]],fa)
split=replace(overload,max_vehicles=3)
assert check_temporal_equations(split,[[1],[2]],check_cvrptw(split,[[1],[2]]))
import copy
corrupt=copy.deepcopy(fa);corrupt["schedules"][0].loc[0,"arrival"]+=.5
try: check_temporal_equations(fixture,[[1,2]],corrupt)
except AssertionError: pass
else: raise AssertionError("Temporal equation checker accepted corrupt arrival")
print("PASS: synthetic window/wait/service/depot/optional-field/shape/zero/asymmetric/fleet/capacity and full-arc equation checks.")
# A real supplied route's malformed variants, only if real data were checked.
if reference_audits:
    inst,routes,audit=reference_audits["C101"]
    assert not check_cvrptw(inst,routes[:-1])["feasible"]
    assert not check_cvrptw(inst,routes+[routes[0]])["feasible"]
else: print("Real-reference negative cases: NOT RUN")

# Synthetic timeline is meaningful even when no benchmark collection is available.
display(ta["schedules"][0])
fig, ax = plt.subplots(figsize=(9, 2.8))
row=ta["schedules"][0].iloc[0]
ax.broken_barh([(0,row.arrival)],(0,1),facecolors="steelblue",label="Outbound travel")
ax.broken_barh([(row.arrival,row.waiting)],(0,1),facecolors="darkorange",label="Waiting")
ax.broken_barh([(row.start,row.departure-row.start)],(0,1),facecolors="seagreen",label="Service")
ax.broken_barh([(row.departure,1)],(0,1),facecolors="steelblue")
ax.axvline(5,color="black",linestyle="--",label="Required service start = 5")
ax.axvline(10,color="crimson",linestyle=":",label="Depot closes = 10")
ax.set(xlim=(0,11),ylim=(-.2,1.2),yticks=[],xlabel="Elapsed synthetic time units",title="Supplied synthetic route: wait, start on time, finish after customer close, return by depot close")
ax.legend(loc="upper center",bbox_to_anchor=(.5,-.28),ncol=3);plt.tight_layout();plt.show()


## 11 · Review checkpoint and remaining decisions

Review the distance objective, coverage, flow, capacity, at-most-K fleet rule, subtour elimination, service-start windows, waiting, and depot return with the tutor. The active convention is hard service-start windows: early arrival waits, and a used vehicle returns by depot closing.

Algorithm 1 (ALNS + VND) is implemented, tuned on a development split and evaluated on held-out instances in Sections 12–15. Final method selection, the PyVRP baseline, ILS/HGS and an agreed project runtime budget remain open. The brief requires at least 20 distinct evaluated instances and compatible reference comparisons. Supplied-route checks here do not satisfy that experimental requirement.

## 12 · Algorithm 1 — ALNS + VND: how do we actually *find* good routes?

Sections 1–11 define **what** a feasible, minimum-distance plan is. The exact MILP has $K(n+1)^2$ binary variables, which is hopeless at 1,000 customers. This part of the notebook contains our first **metaheuristic**: an *Adaptive Large Neighbourhood Search* (ALNS, Ropke & Pisinger 2006) whose every candidate solution is polished by a *Variable Neighbourhood Descent* (VND, Hansen & Mladenović 2001).

| Ingredient | Role | Where |
| --- | --- | --- |
| Solution state | routes as linked lists, always capacity- and time-window feasible, at most $K$ routes by construction | §13.1–13.3 |
| O(1) time-window test | prefix/suffix *segments* (duration, time warp, earliest, latest start) per customer | §13.2 |
| VND | relocate, swap, 2-opt\*, or-opt (2–3, reversible), 2-opt on granular neighbour lists | §13.4–13.5 |
| ALNS destroy / repair | random, Shaw, SISR string and route removal; greedy, regret-2/3 and blink insertion | §13.6–13.7 |
| Guidance | adaptive operator weights, simulated-annealing acceptance on the time budget | §13.8 |
| Driver | preprocessing, time management, result packaging | §14 |

**The notebook is the authoritative source of the solver.** Every cell headed `%%alns_source <module> <part>` is solver code. Running such a cell only *records* it; `load_alns()` (next cell) then assembles the four modules `_jit`, `config`, `kernels`, `solver`:

* if they are identical to `algorithms/alns_vnd/*.py` (apart from blank lines), the **precompiled** copy is used — Numba keeps compiled machine code on disk only for real files, so the live demo starts in about a second instead of 30–60 s;
* if a cell was edited, the notebook's **own** code is compiled instead (with a warning). `python scripts/sync_notebook_solver.py --write` then refreshes the `.py` copy, which the isolated benchmark workers and the unit tests import; `tests/test_notebook_solver.py` fails while the two differ.

The kernels are plain Python in the Numba-compatible subset. Numba (BSD licence) only compiles them; without it the same code runs as ordinary, much slower Python.

In [ ]:
# ── Solver source management ────────────────────────────────────────────────
# %%alns_source cells record solver code; load_alns() assembles and loads it.
import importlib
import time
import types

from IPython.core.magic import register_cell_magic

ALNS_MODULES = ("_jit", "config", "kernels", "solver")
ALNS_PACKAGE_DIR = PROJECT_ROOT / "algorithms" / "alns_vnd"
_ALNS_PARTS = {name: {} for name in ALNS_MODULES}


@register_cell_magic
def alns_source(line, cell):
    """Record one part of a solver module. Header: %%alns_source <module> <part>."""
    module, part = line.split()
    if module not in _ALNS_PARTS:
        raise ValueError(f"Unknown solver module {module!r}")
    compile(cell, f"<main.ipynb:{module}:{part}>", "exec")  # syntax check immediately
    _ALNS_PARTS[module][int(part)] = cell                   # re-running a cell replaces it


def _alns_text(module):
    parts = _ALNS_PARTS[module]
    return "\n\n\n".join(parts[i].strip("\n") for i in sorted(parts)) + "\n"


def _normalized(text):
    """Blank lines and trailing spaces do not change a Python program."""
    return [line.rstrip() for line in text.splitlines() if line.strip()]


def load_alns(force_notebook=False):
    """Return (package, mode). Uses the precompiled .py copy only if it is identical."""
    missing = [m for m in ALNS_MODULES if not _ALNS_PARTS[m]]
    if missing:
        raise RuntimeError(f"Run the %%alns_source cells first (missing {missing})")
    texts = {m: _alns_text(m) for m in ALNS_MODULES}
    differs = [m for m in ALNS_MODULES
               if _normalized(texts[m]) != _normalized((ALNS_PACKAGE_DIR / f"{m}.py").read_text())]
    if not differs and not force_notebook:
        package = importlib.import_module("algorithms.alns_vnd")
        return package, "precompiled algorithms/alns_vnd (verified identical to these cells)"
    # Compile the notebook's own code as a fresh package named alns_notebook.
    for name in [k for k in sys.modules if k == "alns_notebook" or k.startswith("alns_notebook.")]:
        del sys.modules[name]
    package = types.ModuleType("alns_notebook")
    package.__path__ = []
    sys.modules["alns_notebook"] = package
    for m in ALNS_MODULES:
        module = types.ModuleType(f"alns_notebook.{m}")
        module.__package__ = "alns_notebook"
        sys.modules[module.__name__] = module
        setattr(package, m, module)
        exec(compile(texts[m], f"<main.ipynb alns_vnd/{m}.py>", "exec"), module.__dict__)
    package.solver.NOTEBOOK_SOURCE_SHA256 = {
        f"{m}.py": hashlib.sha256(texts[m].encode()).hexdigest() for m in ALNS_MODULES}
    package.solve, package.ALNSConfig = package.solver.solve, package.config.ALNSConfig
    package.VERSION, package.warmup = package.solver.VERSION, package.solver.warmup
    reason = f"cells differ from .py in {differs}" if differs else "forced by ECOROUTE_ALNS_FROM_NOTEBOOK=1"
    warnings.warn(f"Compiling the solver from notebook cells ({reason}); expect 30-60 s of Numba "
                  "compilation. Run `python scripts/sync_notebook_solver.py --write` to refresh the "
                  "precompiled copy used by tests and benchmark workers.")
    return package, f"compiled from notebook cells ({reason})"


print("Solver cells are recorded by %%alns_source; load them with load_alns().")

### 12.1 · Optional JIT compilation (`_jit`)

A single decorator decides how kernels run: compiled by Numba with an on-disk cache when the code lives in a file, compiled without cache when it comes from these cells, or as plain Python when Numba is absent or `ECOROUTE_DISABLE_JIT=1`. `error_model="numpy"` removes Python's division-by-zero checks from the hot loops.

In [ ]:
%%alns_source _jit 1
"""Optional Numba JIT.

The kernels are ordinary Python functions written in the Numba-compatible
subset.  When Numba is installed they are compiled to machine code (and cached
on disk, so later processes skip compilation); otherwise, or with
``ECOROUTE_DISABLE_JIT=1``, the very same source runs as plain (slow) Python.
"""
import os

JIT_ENABLED = False
if os.environ.get("ECOROUTE_DISABLE_JIT", "") != "1":
    try:
        import numba  # noqa: F401
        JIT_ENABLED = True
    except ImportError:  # pragma: no cover - exercised only without numba
        JIT_ENABLED = False

if JIT_ENABLED:
    from numba import njit as _numba_njit

    def _cacheable(function):
        # Numba can only cache code that lives in a real source file; the same
        # functions compiled from notebook cells are JIT-compiled without cache.
        return os.path.isfile(function.__code__.co_filename)

    def njit(function=None, inline=False):
        # inline=True inlines at Numba-IR level: removes call overhead (array
        # reference counting) for tiny hot helpers.
        if function is None:
            return lambda f: njit(f, inline)
        cache = _cacheable(function)
        if inline:
            return _numba_njit(cache=cache, error_model="numpy", inline="always")(function)
        # cache=True writes compiled machine code beside this package so that a
        # freshly spawned benchmark worker does not pay compilation inside its
        # time budget; error_model='numpy' avoids Python-style ZeroDivision checks.
        return _numba_njit(cache=cache, error_model="numpy")(function)
else:
    def njit(function=None, inline=False):
        if function is None:
            return lambda f: f
        return function

### 12.2 · Every parameter is configurable (`config`)

All algorithmic settings live in one frozen dataclass; unknown keys are rejected so that a typo in an experiment plan cannot silently fall back to a default. The defaults are the values **frozen after tuning on the development split only** (§15.4): annealing temperature $1.5 \rightarrow 0.02$ × mean arc length, 20 VND neighbours, worst removal disabled. `clock="process"` makes the isolated benchmark worker charge interpreter start-up to the budget, as the evaluator does.

In [ ]:
%%alns_source config 1
"""Configuration of the ALNS + VND solver.

Every algorithmic setting is a field here; ``ALNSConfig.from_dict`` rejects
unknown keys so that a typo in an experiment plan cannot silently fall back to a
default.  Defaults are the frozen tuned values (dev campaigns dev-001..dev-007,
see results/alns-vnd/README.md).
"""
from dataclasses import asdict, dataclass, fields
import math

DESTROY_OPERATORS = ("random", "worst", "shaw", "string", "route")
REPAIR_OPERATORS = ("greedy", "regret2", "regret3", "blink")
NEIGHBOURHOODS = ("relocate", "swap", "two_opt_star", "or_opt", "two_opt")


@dataclass(frozen=True)
class ALNSConfig:
    # --- stopping ---------------------------------------------------------
    # Fraction of the wall-clock limit the search may use, and an absolute
    # reserve (seconds) for returning the result.  clock="call" measures the
    # budget from the solve() call; clock="process" measures it from the start
    # of the current process (needs psutil), which is what the isolated
    # benchmark worker enforces: interpreter start-up and imports are charged.
    clock: str = "call"
    time_fraction: float = 0.97
    time_reserve: float = 0.75
    max_iterations: int = 0            # 0 = unlimited (time-limited only)
    chunk_seconds: float = 0.05        # granularity of time checks

    # --- granular neighbourhoods -------------------------------------------
    n_neighbours: int = 40             # repair: routes of these neighbours are candidates
    vnd_neighbours: int = 20           # VND: neighbours v examined per customer u
    near_list: int = 100               # SISR adjacency length
    shaw_list: int = 100               # Shaw relatedness list length
    tw_wait_weight: float = 0.2        # proximity: penalty on forced waiting
    tw_warp_weight: float = 1.0        # proximity: penalty on lateness
    insert_all_routes: bool = False    # repair: scan every route (slow, n<=200)

    # --- destroy ------------------------------------------------------------
    destroy_min: int = 5
    destroy_max: int = 40
    destroy_max_fraction: float = 0.2  # q_max = min(destroy_max, frac * n)
    worst_randomness: float = 3.0      # p in y^p (Ropke & Pisinger)
    shaw_randomness: float = 6.0
    shaw_weights: tuple = (9.0, 3.0, 2.0)  # distance, time window, demand
    sisr_avg_removed: float = 10.0     # c-bar
    sisr_max_string: int = 10          # L_max
    sisr_split_rate: float = 0.5
    sisr_split_depth: float = 0.01     # beta
    blink_rate: float = 0.01

    # --- adaptive operator selection ---------------------------------------
    destroy_weights: tuple = (1.0, 0.0, 1.0, 1.0, 0.5)   # 0 disables (worst off: tuned)
    repair_weights: tuple = (1.0, 1.0, 1.0, 1.0)
    score_new_best: float = 33.0
    score_better: float = 9.0
    score_accepted: float = 13.0
    reaction: float = 0.1
    segment: int = 100

    # --- acceptance / diversification ---------------------------------------
    # Simulated annealing with temperatures relative to the mean arc length of
    # the initial solution, cooled exponentially over the elapsed budget.
    temp_start: float = 1.5
    temp_end: float = 0.02
    restart_after: int = 0             # iterations without best improvement; 0 = off

    # --- VND ----------------------------------------------------------------
    neighbourhoods: tuple = NEIGHBOURHOODS

    def __post_init__(self):
        def positive(name, integer=False, allow_zero=False):
            value = getattr(self, name)
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                raise ValueError(f"{name} must be numeric")
            if integer and int(value) != value:
                raise ValueError(f"{name} must be an integer")
            if not math.isfinite(value) or value < 0 or (value == 0 and not allow_zero):
                raise ValueError(f"{name} must be {'nonnegative' if allow_zero else 'positive'}")
        for name in ("time_fraction", "chunk_seconds", "destroy_max_fraction", "worst_randomness",
                     "shaw_randomness", "sisr_avg_removed", "temp_start", "temp_end"):
            positive(name)
        for name in ("time_reserve", "tw_wait_weight", "tw_warp_weight", "blink_rate", "score_new_best",
                     "score_better", "score_accepted", "sisr_split_rate", "sisr_split_depth"):
            positive(name, allow_zero=True)
        for name in ("n_neighbours", "vnd_neighbours", "near_list", "shaw_list", "destroy_min", "destroy_max",
                     "sisr_max_string"):
            positive(name, integer=True)
        for name in ("max_iterations", "segment", "restart_after"):
            positive(name, integer=True, allow_zero=True)
        if self.clock not in ("call", "process"):
            raise ValueError("clock must be 'call' or 'process'")
        if self.time_fraction > 1:
            raise ValueError("time_fraction must be <= 1")
        if not 0 < self.reaction <= 1:
            raise ValueError("reaction must be in (0, 1]")
        if self.destroy_min > self.destroy_max:
            raise ValueError("destroy_min must not exceed destroy_max")
        if self.temp_end > self.temp_start:
            raise ValueError("temp_end must not exceed temp_start")
        if not 0 <= self.blink_rate < 1 or not 0 <= self.sisr_split_rate <= 1 or not 0 <= self.sisr_split_depth < 1:
            raise ValueError("probabilities out of range")
        for name, names in (("destroy_weights", DESTROY_OPERATORS), ("repair_weights", REPAIR_OPERATORS)):
            weights = getattr(self, name)
            if len(weights) != len(names) or any(not math.isfinite(w) or w < 0 for w in weights) or sum(weights) <= 0:
                raise ValueError(f"{name} needs {len(names)} nonnegative weights, not all zero ({names})")
        if len(self.shaw_weights) != 3 or any(w < 0 for w in self.shaw_weights):
            raise ValueError("shaw_weights = (distance, time, demand) nonnegative")
        if not self.neighbourhoods or any(nb not in NEIGHBOURHOODS for nb in self.neighbourhoods) \
                or len(set(self.neighbourhoods)) != len(self.neighbourhoods):
            raise ValueError(f"neighbourhoods must be distinct names from {NEIGHBOURHOODS}")

    @classmethod
    def from_dict(cls, values=None):
        values = dict(values or {})
        known = {f.name for f in fields(cls)}
        unknown = set(values) - known
        if unknown:
            raise ValueError(f"Unknown ALNS configuration keys: {sorted(unknown)}")
        for key in ("shaw_weights", "destroy_weights", "repair_weights", "neighbourhoods"):
            if key in values:
                values[key] = tuple(values[key])
        return cls(**values)

    def to_dict(self):
        return {k: list(v) if isinstance(v, tuple) else v for k, v in asdict(self).items()}

## 13 · The search kernels (`kernels`)

### 13.1 · Data layout

A solution is a handful of NumPy arrays: per-customer links (`NXT`, `PRV`), route id and position; per-customer prefix/suffix time-window segments and cumulative load; per-route first/last customer, length, load, cost and a *modification stamp*. Unassigned customers have route id −1. Depot 0 is a sentinel at both ends of every route, and at most $R=\min(K,n)$ route slots exist, so the fleet limit can never be exceeded.

In [ ]:
%%alns_source kernels 1
"""Numba kernels of the ALNS + VND metaheuristic for static CVRPTW.

Everything performance-critical lives here: the solution data structure, O(1)
time-window feasibility tests, destroy/repair operators, the Variable
Neighbourhood Descent and the ALNS main loop.  The code is plain Python written
in the Numba-compatible subset (see ``_jit.py``).

Conventions
-----------
* Node 0 is the depot, customers are 1..n (the internal indices of
  ``CVRPTWInstance``).  ``N = n + 1``.
* A *route* is a doubly linked list of customers (``NXT``/``PRV``); the depot
  is represented by the sentinel value 0 at both ends.  At most ``R = min(K, n)``
  routes exist; an unused route has ``LEN == 0``.
* Time windows constrain the **start of service**.  Feasibility is evaluated
  with the time-window segment algebra of Vidal et al. (2013): a sequence of
  visits is summarised by (D, TW, E, L) = (minimum duration, time warp,
  earliest and latest start).  Concatenation is O(1) and a sequence is
  feasible iff its time warp is zero.  Every route keeps prefix segments
  (depot .. u) and suffix segments (u .. depot) at each node, so inserting,
  removing or exchanging customers/segments between routes is checked in O(1).
* Search states are always capacity- and time-window feasible.  Customers that
  cannot be inserted stay *unassigned* and are penalised in the objective; a
  state with unassigned customers is never reported as a solution.

Data passed to the kernels
--------------------------
Instance tuple ``inst = (dist, tt, ND, cap, nbrs, near, shaw, sym, empty_cost)``
where ``ND`` stacks per-node rows (TA, TB, SVC, DEM).
Solution tuple ``sol = (I, F, RI, RF, TST, SC, CTR)`` with the row layouts
below; ``CTR`` (a 1-element modification counter) is shared by all copies.

Performance note: handing these *tuples* to a non-inlined Numba function costs
~100 ns per call (reference counting of every member array), while handing
the arrays individually is essentially free.  Only the per-iteration entry
points therefore accept tuples; every function on a hot path takes arrays.
"""
import numpy as np

from ._jit import njit

# ---------------------------------------------------------------- layout --
# ND (float64, 4 x N): per-node instance data
TA = 0    # earliest service start
TB = 1    # latest service start
SVC = 2   # service duration
DEM = 3   # demand
# I  (int64,   4 x N): per-node links and positions
NXT = 0
PRV = 1
RTE = 2   # route id, -1 when unassigned
POS = 3   # 0-based position inside its route
# F  (float64, 10 x N): per-node segment data of its current route
PD = 0    # prefix segment depot..u : duration
PTW = 1   #                          time warp
PE = 2    #                          earliest start
PL = 3    #                          latest start
SD = 4    # suffix segment u..depot : duration
STW = 5
SE = 6
SL = 7
PLOAD = 8  # cumulative load depot..u
ES = 9     # earliest service start at u when leaving the depot at a0
# RI (int64,   4 x R): per-route integers
FIRST = 0
LAST = 1
LEN = 2
STAMP = 3  # value of CTR when the route last changed
# RF (float64, 2 x R): per-route floats
LOAD = 0
COST = 1
# SC (float64, 2): [routing distance, number of unassigned customers]

# VND neighbourhood identifiers (order configurable from Python)
NB_RELOCATE = 0
NB_SWAP = 1
NB_TWO_OPT_STAR = 2
NB_OR_OPT = 3
NB_TWO_OPT = 4
N_NEIGHBOURHOODS = 5

# destroy operators
D_RANDOM = 0
D_WORST = 1
D_SHAW = 2
D_STRING = 3
D_ROUTE = 4
N_DESTROY = 5
# repair operators
R_GREEDY = 0
R_REGRET2 = 1
R_REGRET3 = 2
R_BLINK = 3
N_REPAIR = 4

# integer parameter vector IP
IP_QMIN = 0
IP_QMAX = 1
IP_SEGMENT = 2
IP_RESTART = 3
IP_SISR_LMAX = 4
IP_ITER = 5            # global iteration counter (state)
IP_LAST_IMPROVE = 6    # iteration of the last best improvement (state)
IP_INSERT_ALL = 7      # 1: repair scans every route, 0: granular candidate routes
IP_VND_NEIGHBOURS = 8  # neighbours examined per customer in the VND
N_IP = 9
# float parameter vector P
P_WORST_P = 0
P_SHAW_P = 1
P_SISR_CBAR = 2
P_SISR_SPLIT = 3
P_SISR_BETA = 4
P_BLINK = 5
P_SIG1 = 6
P_SIG2 = 7
P_SIG3 = 8
P_REACTION = 9
P_PENALTY = 10
N_P = 11
# statistics vector ST (float64)
ST_ITERS = 0
ST_ACCEPT = 1
ST_IMPROVE_CUR = 2
ST_NEW_BEST = 3
ST_RESTARTS = 4
ST_INFEASIBLE_CAND = 5
ST_VND_MOVES = 6
N_ST = 7

EPS_TW = 1e-9     # time-warp tolerance (validator uses 1e-7, so we are stricter)
EPS_COST = 1e-7   # minimum strict improvement
EPS_LOAD = 1e-9
INF = 1e30
BIG_REGRET = 1e12

### 13.2 · Constant-time time-window feasibility

Following Vidal et al. (2013), a sequence of visits is summarised by four numbers: minimum duration $D$, time warp $TW$ (lateness that would be needed), earliest start $E$ and latest start $L$. Two sequences joined by an arc of travel time $t$ combine in O(1):

$$\Delta=D_1-TW_1+t,\quad TW=TW_1+TW_2+\max(E_1+\Delta-L_2,0),\quad D=D_1+D_2+t+\max(E_2-\Delta-L_1,0).$$

A route is feasible iff its time warp is 0. The depot segment $[a_0,H]$ lets the departure time float inside the depot window, and windows constrain the **start** of service exactly as in TW2 of the MILP. With prefix and suffix segments stored at every customer, inserting a customer or splicing route pieces is checked without walking the route.

In [ ]:
%%alns_source kernels 2
# ------------------------------------------------------- segment algebra --
@njit
def cat(D1, W1, E1, L1, D2, W2, E2, L2, t):
    """Concatenate two time-window segments separated by travel time t."""
    delta = D1 - W1 + t
    dwt = E2 - delta - L1
    if dwt < 0.0:
        dwt = 0.0
    dtw = E1 + delta - L2
    if dtw < 0.0:
        dtw = 0.0
    E = E2 - delta
    if E1 > E:
        E = E1
    L = L2 - delta
    if L1 < L:
        L = L1
    return D1 + D2 + t + dwt, W1 + W2 + dtw, E - dwt, L + dtw


@njit
def pre(F, ND, x):
    """Prefix segment depot..x (x == 0: the bare start depot)."""
    if x == 0:
        return 0.0, 0.0, ND[TA, 0], ND[TB, 0]
    return F[PD, x], F[PTW, x], F[PE, x], F[PL, x]


@njit
def suf(F, ND, x):
    """Suffix segment x..depot (x == 0: the bare end depot)."""
    if x == 0:
        return 0.0, 0.0, ND[TA, 0], ND[TB, 0]
    return F[SD, x], F[STW, x], F[SE, x], F[SL, x]


@njit
def pload(F, x):
    if x == 0:
        return 0.0
    return F[PLOAD, x]


@njit
def join_tw(tt, ND, F, p, x):
    """Time warp of prefix(p) + suffix(x): the route after deleting what lies between."""
    D1, W1, E1, L1 = pre(F, ND, p)
    D2, W2, E2, L2 = suf(F, ND, x)
    return cat(D1, W1, E1, L1, D2, W2, E2, L2, tt[p, x])[1]


@njit
def insert_tw(tt, ND, F, p, u, x):
    """Time warp of prefix(p) + u + suffix(x)."""
    D, W, E, L = pre(F, ND, p)
    D, W, E, L = cat(D, W, E, L, ND[SVC, u], 0.0, ND[TA, u], ND[TB, u], tt[p, u])
    D2, W2, E2, L2 = suf(F, ND, x)
    return cat(D, W, E, L, D2, W2, E2, L2, tt[u, x])[1]


@njit
def chain_seg(dist, tt, ND, I, first, length, reverse):
    """Segment of ``length`` consecutive nodes starting at ``first`` (following
    NXT), optionally visited in reverse order.  Returns (D, TW, E, L, cost)
    where cost is the internal travel distance in the visiting direction."""
    u = first
    if reverse:
        for k in range(length - 1):
            u = I[NXT, u]
    D, W, E, L = ND[SVC, u], 0.0, ND[TA, u], ND[TB, u]
    cost = 0.0
    for k in range(length - 1):
        v = I[PRV, u] if reverse else I[NXT, u]
        D, W, E, L = cat(D, W, E, L, ND[SVC, v], 0.0, ND[TA, v], ND[TB, v], tt[u, v])
        cost += dist[u, v]
        u = v
    return D, W, E, L, cost

### 13.3 · Keeping route data consistent

After every applied move only the affected routes are recomputed (positions, loads, cost, prefix/suffix segments, earliest service starts) and their stamp is advanced. `seq_feasible` is an independent earliest-schedule check used for intra-route moves.

In [ ]:
%%alns_source kernels 3
# ------------------------------------------------------ route maintenance --
@njit
def update_route(dist, tt, ND, I, F, RI, RF, CTR, r):
    """Recompute positions, loads, cost and prefix/suffix segments of route r."""
    D, W, E, L = 0.0, 0.0, ND[TA, 0], ND[TB, 0]
    t = ND[TA, 0]
    last = 0
    load = 0.0
    cost = 0.0
    k = 0
    u = RI[FIRST, r]
    while u != 0:
        D, W, E, L = cat(D, W, E, L, ND[SVC, u], 0.0, ND[TA, u], ND[TB, u], tt[last, u])
        F[PD, u] = D
        F[PTW, u] = W
        F[PE, u] = E
        F[PL, u] = L
        t = t + tt[last, u]
        if t < ND[TA, u]:
            t = ND[TA, u]
        F[ES, u] = t
        t += ND[SVC, u]
        load += ND[DEM, u]
        F[PLOAD, u] = load
        cost += dist[last, u]
        I[RTE, u] = r
        I[POS, u] = k
        k += 1
        last = u
        u = I[NXT, u]
    RI[LAST, r] = last
    RI[LEN, r] = k
    RF[LOAD, r] = load
    if k == 0:
        RI[FIRST, r] = 0
        RF[COST, r] = 0.0
    else:
        RF[COST, r] = cost + dist[last, 0]
    D, W, E, L = 0.0, 0.0, ND[TA, 0], ND[TB, 0]
    nxt = 0
    u = last
    while u != 0:
        D, W, E, L = cat(ND[SVC, u], 0.0, ND[TA, u], ND[TB, u], D, W, E, L, tt[u, nxt])
        F[SD, u] = D
        F[STW, u] = W
        F[SE, u] = E
        F[SL, u] = L
        nxt = u
        u = I[PRV, u]
    CTR[0] += 1
    RI[STAMP, r] = CTR[0]


@njit
def set_route(dist, tt, ND, I, F, RI, RF, CTR, r, seq, m):
    """Make route r visit seq[0..m-1] in order."""
    prev = 0
    for k in range(m):
        u = seq[k]
        I[PRV, u] = prev
        if prev != 0:
            I[NXT, prev] = u
        prev = u
    if m > 0:
        I[NXT, prev] = 0
        RI[FIRST, r] = seq[0]
    else:
        RI[FIRST, r] = 0
    update_route(dist, tt, ND, I, F, RI, RF, CTR, r)


@njit
def collect(I, RI, r, buf):
    m = 0
    u = RI[FIRST, r]
    while u != 0:
        buf[m] = u
        m += 1
        u = I[NXT, u]
    return m


@njit
def unlink(I, RI, u):
    """Detach u from its route (route data become stale until update_route)."""
    r = I[RTE, u]
    p = I[PRV, u]
    x = I[NXT, u]
    if p == 0:
        RI[FIRST, r] = x
    else:
        I[NXT, p] = x
    if x != 0:
        I[PRV, x] = p
    I[RTE, u] = -1
    I[NXT, u] = 0
    I[PRV, u] = 0


@njit
def link_after(I, RI, u, v, r):
    """Insert detached u directly after v in route r (v == 0: at the start)."""
    if v == 0:
        x = RI[FIRST, r]
        RI[FIRST, r] = u
    else:
        x = I[NXT, v]
        I[NXT, v] = u
    I[PRV, u] = v
    I[NXT, u] = x
    if x != 0:
        I[PRV, x] = u
    I[RTE, u] = r


@njit
def seq_feasible(tt, ND, seq, m):
    """Earliest-schedule feasibility of one route visiting seq[0..m-1]."""
    t = ND[TA, 0]
    prev = 0
    for k in range(m):
        u = seq[k]
        t = t + tt[prev, u]
        if t < ND[TA, u]:
            t = ND[TA, u]
        if t > ND[TB, u] + EPS_TW:
            return False
        t += ND[SVC, u]
        prev = u
    return t + tt[prev, 0] <= ND[TB, 0] + EPS_TW


@njit
def seq_cost(dist, seq, m):
    if m == 0:
        return 0.0
    c = dist[0, seq[0]]
    for k in range(1, m):
        c += dist[seq[k - 1], seq[k]]
    return c + dist[seq[m - 1], 0]


@njit
def total_cost(RI, RF):
    c = 0.0
    for r in range(RI.shape[1]):
        if RI[LEN, r] > 0:
            c += RF[COST, r]
    return c


@njit
def refresh_score(I, RI, RF, SC):
    SC[0] = total_cost(RI, RF)
    un = 0
    for u in range(1, I.shape[1]):
        if I[RTE, u] < 0:
            un += 1
    SC[1] = un


@njit
def objective(SC, penalty):
    return SC[0] + penalty * SC[1]


@njit
def copy_sol(src, dst):
    dst[0][:, :] = src[0]
    dst[1][:, :] = src[1]
    dst[2][:, :] = src[2]
    dst[3][:, :] = src[3]
    dst[4][:, :] = src[4]
    dst[5][:] = src[5]


@njit
def used_routes(RI):
    k = 0
    for r in range(RI.shape[1]):
        if RI[LEN, r] > 0:
            k += 1
    return k


@njit
def empty_route(RI):
    for r in range(RI.shape[1]):
        if RI[LEN, r] == 0:
            return r
    return -1


@njit
def seed_rng(seed):
    np.random.seed(seed)

### 13.4 · VND neighbourhoods

Each move first computes its distance change from the arcs it removes and adds; only strictly improving moves (by more than $10^{-7}$) are checked for capacity and time windows. Moves between two routes use the O(1) segment test; moves inside one route rebuild the route and verify it exactly. A move is applied only if it is feasible **and** improving. `tests/test_alns_vnd.py` compares every move with a brute-force reference judged by the shared validator.

In [ ]:
%%alns_source kernels 4
# ======================================================== VND moves =======
# All move functions evaluate the distance change first (O(1)), then capacity,
# then time windows: O(1) segment concatenation between routes, an exact
# O(route length) rebuild for intra-route moves.  They apply the move and
# return True only if it is feasible and strictly improving.

@njit
def try_relocate(dist, tt, ND, cap, I, F, RI, RF, CTR, u, v, rv, buf):
    """Move u directly after v in route rv (v == 0: to the start of rv)."""
    ru = I[RTE, u]
    x = RI[FIRST, rv] if v == 0 else I[NXT, v]
    if v == u or x == u:
        return False
    pu = I[PRV, u]
    nu = I[NXT, u]
    delta = (dist[pu, nu] - dist[pu, u] - dist[u, nu]
             + dist[v, u] + dist[u, x] - dist[v, x])
    if delta > -EPS_COST:
        return False
    if ru != rv:
        if RF[LOAD, rv] + ND[DEM, u] > cap + EPS_LOAD:
            return False
        if insert_tw(tt, ND, F, v, u, x) > EPS_TW:
            return False
        if join_tw(tt, ND, F, pu, nu) > EPS_TW:
            return False
        unlink(I, RI, u)
        link_after(I, RI, u, v, rv)
        update_route(dist, tt, ND, I, F, RI, RF, CTR, ru)
        update_route(dist, tt, ND, I, F, RI, RF, CTR, rv)
        return True
    # intra-route: build the new order and verify it from scratch
    m = 0
    if v == 0:
        buf[m] = u
        m += 1
    y = RI[FIRST, ru]
    while y != 0:
        if y != u:
            buf[m] = y
            m += 1
            if y == v:
                buf[m] = u
                m += 1
        y = I[NXT, y]
    if not seq_feasible(tt, ND, buf, m):
        return False
    if seq_cost(dist, buf, m) > RF[COST, ru] - EPS_COST:
        return False
    set_route(dist, tt, ND, I, F, RI, RF, CTR, ru, buf, m)
    return True


@njit
def try_relocate_new_route(dist, tt, ND, I, F, RI, RF, CTR, empty_cost, u):
    """Move u alone into an unused route when that shortens the plan."""
    if empty_cost[u] >= INF:
        return False
    pu = I[PRV, u]
    nu = I[NXT, u]
    if pu == 0 and nu == 0:
        return False
    delta = dist[pu, nu] - dist[pu, u] - dist[u, nu] + empty_cost[u]
    if delta > -EPS_COST:
        return False
    r = empty_route(RI)
    if r < 0:
        return False
    if join_tw(tt, ND, F, pu, nu) > EPS_TW:
        return False
    ru = I[RTE, u]
    unlink(I, RI, u)
    link_after(I, RI, u, 0, r)
    update_route(dist, tt, ND, I, F, RI, RF, CTR, ru)
    update_route(dist, tt, ND, I, F, RI, RF, CTR, r)
    return True


@njit
def try_swap(dist, tt, ND, cap, I, F, RI, RF, CTR, u, v, buf):
    ru = I[RTE, u]
    rv = I[RTE, v]
    pu = I[PRV, u]
    nu = I[NXT, u]
    pv = I[PRV, v]
    nv = I[NXT, v]
    if ru != rv:
        delta = (dist[pu, v] + dist[v, nu] - dist[pu, u] - dist[u, nu]
                 + dist[pv, u] + dist[u, nv] - dist[pv, v] - dist[v, nv])
        if delta > -EPS_COST:
            return False
        if RF[LOAD, ru] - ND[DEM, u] + ND[DEM, v] > cap + EPS_LOAD:
            return False
        if RF[LOAD, rv] - ND[DEM, v] + ND[DEM, u] > cap + EPS_LOAD:
            return False
        if insert_tw(tt, ND, F, pu, v, nu) > EPS_TW:
            return False
        if insert_tw(tt, ND, F, pv, u, nv) > EPS_TW:
            return False
        I[PRV, v] = pu
        I[NXT, v] = nu
        if pu == 0:
            RI[FIRST, ru] = v
        else:
            I[NXT, pu] = v
        if nu != 0:
            I[PRV, nu] = v
        I[PRV, u] = pv
        I[NXT, u] = nv
        if pv == 0:
            RI[FIRST, rv] = u
        else:
            I[NXT, pv] = u
        if nv != 0:
            I[PRV, nv] = u
        update_route(dist, tt, ND, I, F, RI, RF, CTR, ru)
        update_route(dist, tt, ND, I, F, RI, RF, CTR, rv)
        return True
    if nu == v:
        delta = (dist[pu, v] + dist[v, u] + dist[u, nv]
                 - dist[pu, u] - dist[u, v] - dist[v, nv])
    elif nv == u:
        delta = (dist[pv, u] + dist[u, v] + dist[v, nu]
                 - dist[pv, v] - dist[v, u] - dist[u, nu])
    else:
        delta = (dist[pu, v] + dist[v, nu] - dist[pu, u] - dist[u, nu]
                 + dist[pv, u] + dist[u, nv] - dist[pv, v] - dist[v, nv])
    if delta > -EPS_COST:
        return False
    m = 0
    y = RI[FIRST, ru]
    while y != 0:
        if y == u:
            buf[m] = v
        elif y == v:
            buf[m] = u
        else:
            buf[m] = y
        m += 1
        y = I[NXT, y]
    if not seq_feasible(tt, ND, buf, m):
        return False
    if seq_cost(dist, buf, m) > RF[COST, ru] - EPS_COST:
        return False
    set_route(dist, tt, ND, I, F, RI, RF, CTR, ru, buf, m)
    return True


@njit
def try_two_opt_star(dist, tt, ND, cap, I, F, RI, RF, CTR, u, v):
    """Inter-route tail exchange creating arc (u, v):
    route(u) := depot..u + v..depot,  route(v) := depot..prev(v) + next(u)..depot."""
    ru = I[RTE, u]
    rv = I[RTE, v]
    if ru == rv:
        return False
    pv = I[PRV, v]
    nu = I[NXT, u]
    delta = dist[u, v] + dist[pv, nu] - dist[u, nu] - dist[pv, v]
    if delta > -EPS_COST:
        return False
    head_u = pload(F, u)
    head_v = pload(F, pv)
    if head_u + (RF[LOAD, rv] - head_v) > cap + EPS_LOAD:
        return False
    if head_v + (RF[LOAD, ru] - head_u) > cap + EPS_LOAD:
        return False
    D1, W1, E1, L1 = pre(F, ND, u)
    D2, W2, E2, L2 = suf(F, ND, v)
    if cat(D1, W1, E1, L1, D2, W2, E2, L2, tt[u, v])[1] > EPS_TW:
        return False
    if join_tw(tt, ND, F, pv, nu) > EPS_TW:
        return False
    I[NXT, u] = v
    I[PRV, v] = u
    if pv == 0:
        RI[FIRST, rv] = nu
    else:
        I[NXT, pv] = nu
    if nu != 0:
        I[PRV, nu] = pv
    update_route(dist, tt, ND, I, F, RI, RF, CTR, ru)
    update_route(dist, tt, ND, I, F, RI, RF, CTR, rv)
    return True


@njit
def try_or_opt(dist, tt, ND, cap, sym, I, F, RI, RF, CTR, u, length, v, reverse, buf):
    """Move the string of ``length`` customers starting at u (following NXT).

    reverse == False: insert the string after v (creates arc v->u).
    reverse == True : insert it reversed before v (creates arc u->v).
    """
    if v == u:
        return False
    ru = I[RTE, u]
    rv = I[RTE, v]
    last = u
    sload = ND[DEM, u]
    for k in range(length - 1):
        last = I[NXT, last]
        if last == 0 or last == v:
            return False
        sload += ND[DEM, last]
    pf = I[PRV, u]
    nl = I[NXT, last]
    # The string goes between a and b.  With v outside the string the removed
    # arcs (pf,u), (last,nl), (a,b) are distinct, so the arc formula below is
    # exact; the only degenerate placements are excluded here.
    if not reverse:
        a = v
        b = I[NXT, v]
        if a == pf:
            return False          # already there
    else:
        a = I[PRV, v]
        b = v
        if b == nl:
            return False          # in-place reversal: covered by 2-opt
    rem = dist[pf, nl] - dist[pf, u] - dist[last, nl]
    if not reverse:
        delta = rem + dist[a, u] + dist[last, b] - dist[a, b]
    else:
        delta = rem + dist[a, last] + dist[u, b] - dist[a, b]
        if sym == 0:
            c_fwd = chain_seg(dist, tt, ND, I, u, length, False)[4]
            c_rev = chain_seg(dist, tt, ND, I, u, length, True)[4]
            delta += c_rev - c_fwd
    if delta > -EPS_COST:
        return False
    if ru != rv:
        if RF[LOAD, rv] + sload > cap + EPS_LOAD:
            return False
        Ds, Ws, Es, Ls, _ = chain_seg(dist, tt, ND, I, u, length, reverse)
        if Ws > EPS_TW:
            return False
        s_first = last if reverse else u
        s_last = u if reverse else last
        D, W, E, L = pre(F, ND, a)
        D, W, E, L = cat(D, W, E, L, Ds, Ws, Es, Ls, tt[a, s_first])
        D2, W2, E2, L2 = suf(F, ND, b)
        if cat(D, W, E, L, D2, W2, E2, L2, tt[s_last, b])[1] > EPS_TW:
            return False
        if join_tw(tt, ND, F, pf, nl) > EPS_TW:
            return False
        # detach the string, then relink it between a and b
        if pf == 0:
            RI[FIRST, ru] = nl
        else:
            I[NXT, pf] = nl
        if nl != 0:
            I[PRV, nl] = pf
        prev = a
        x = s_first
        for k in range(length):
            y = I[PRV, x] if reverse else I[NXT, x]
            link_after(I, RI, x, prev, rv)
            prev = x
            x = y
        update_route(dist, tt, ND, I, F, RI, RF, CTR, ru)
        update_route(dist, tt, ND, I, F, RI, RF, CTR, rv)
        return True
    # intra-route: rebuild the sequence and verify it exactly
    m = 0
    if a == 0:
        m = _emit_string(I, u, last, length, reverse, buf, m)
    y = RI[FIRST, ru]
    while y != 0:
        if y == u:
            y = nl            # skip the string
            continue
        buf[m] = y
        m += 1
        if y == a:
            m = _emit_string(I, u, last, length, reverse, buf, m)
        y = I[NXT, y]
    if not seq_feasible(tt, ND, buf, m):
        return False
    if seq_cost(dist, buf, m) > RF[COST, ru] - EPS_COST:
        return False
    set_route(dist, tt, ND, I, F, RI, RF, CTR, ru, buf, m)
    return True


@njit
def _emit_string(I, u, last, length, reverse, buf, m):
    x = last if reverse else u
    for k in range(length):
        buf[m] = x
        m += 1
        x = I[PRV, x] if reverse else I[NXT, x]
    return m


@njit
def try_two_opt(dist, tt, ND, sym, I, F, RI, RF, CTR, u, v, buf):
    """Intra-route 2-opt creating arc (u, v): reverse the path between them."""
    r = I[RTE, u]
    if I[RTE, v] != r or u == v:
        return False
    if I[POS, u] < I[POS, v]:
        x = u
        y = v
    else:
        x = v
        y = u
    nx = I[NXT, x]
    ny = I[NXT, y]
    if nx == y:
        return False
    if sym == 1:
        delta = dist[x, y] + dist[nx, ny] - dist[x, nx] - dist[y, ny]
        if delta > -EPS_COST:
            return False
    m = collect(I, RI, r, buf)
    i = I[POS, nx]
    j = I[POS, y]
    while i < j:
        tmp = buf[i]
        buf[i] = buf[j]
        buf[j] = tmp
        i += 1
        j -= 1
    if not seq_feasible(tt, ND, buf, m):
        return False
    if seq_cost(dist, buf, m) > RF[COST, r] - EPS_COST:
        return False
    set_route(dist, tt, ND, I, F, RI, RF, CTR, r, buf, m)
    return True

### 13.5 · Variable Neighbourhood Descent

Customers are scanned in random order against their 20 closest *granular* neighbours (distance plus penalties for forced waiting and lateness). Neighbourhoods are swept in order; after a sweep that improves, the descent restarts at the first one, so it ends at a local optimum of all five. A neighbour pair is skipped when neither of its two routes changed since it was last examined, which is why re-optimising after an ALNS move is cheap.

In [ ]:
%%alns_source kernels 5
@njit
def vnd(inst, sol, order, G, buf):
    """Variable Neighbourhood Descent over granular neighbourhoods.

    Neighbourhoods are explored in ``order``; a pass of neighbourhood k over
    every customer u and its first G neighbours v applies every improving move
    found (first improvement).  After a pass with an improvement the descent
    restarts at the first neighbourhood, otherwise it moves on to the next one;
    it stops when a full sweep finds nothing, i.e. at a local optimum of all
    neighbourhoods.

    Modification stamps skip (u, v) pairs whose two routes are unchanged since u
    was last examined in that neighbourhood, so re-optimising a solution after
    an ALNS destroy/repair only re-examines the affected region.
    """
    dist, tt, ND, cap, nbrs, sym, empty_cost = inst[0], inst[1], inst[2], inst[3], inst[4], inst[7], inst[8]
    I, F, RI, RF, TST, SC, CTR = sol[0], sol[1], sol[2], sol[3], sol[4], sol[5], sol[6]
    N = I.shape[1]
    if G > nbrs.shape[1]:
        G = nbrs.shape[1]
    perm = np.random.permutation(N - 1) + 1
    k = 0
    n_moves = 0
    while k < order.shape[0]:
        nb = order[k]
        improved = False
        for idx in range(N - 1):
            u = perm[idx]
            if I[RTE, u] < 0:
                continue
            last_tested = TST[nb, u]
            TST[nb, u] = CTR[0]
            if nb == NB_RELOCATE and RI[STAMP, I[RTE, u]] > last_tested:
                if try_relocate_new_route(dist, tt, ND, I, F, RI, RF, CTR, empty_cost, u):
                    improved = True
                    n_moves += 1
            for jj in range(G):
                v = nbrs[u, jj]
                rv = I[RTE, v]
                if rv < 0:
                    continue
                ru = I[RTE, u]
                if RI[STAMP, ru] <= last_tested and RI[STAMP, rv] <= last_tested:
                    continue
                moved = False
                if nb == NB_RELOCATE:
                    moved = try_relocate(dist, tt, ND, cap, I, F, RI, RF, CTR, u, v, rv, buf)
                    if not moved:
                        moved = try_relocate(dist, tt, ND, cap, I, F, RI, RF, CTR, u, I[PRV, v], rv, buf)
                elif nb == NB_SWAP:
                    moved = try_swap(dist, tt, ND, cap, I, F, RI, RF, CTR, u, v, buf)
                elif nb == NB_TWO_OPT_STAR:
                    if rv != ru:
                        moved = try_two_opt_star(dist, tt, ND, cap, I, F, RI, RF, CTR, u, v)
                        if not moved:
                            moved = try_two_opt_star(dist, tt, ND, cap, I, F, RI, RF, CTR, v, u)
                elif nb == NB_OR_OPT:
                    for length in range(2, 4):
                        moved = try_or_opt(dist, tt, ND, cap, sym, I, F, RI, RF, CTR, u, length, v, False, buf)
                        if moved:
                            break
                        moved = try_or_opt(dist, tt, ND, cap, sym, I, F, RI, RF, CTR, u, length, v, True, buf)
                        if moved:
                            break
                elif nb == NB_TWO_OPT:
                    if rv == ru:
                        moved = try_two_opt(dist, tt, ND, sym, I, F, RI, RF, CTR, u, v, buf)
                if moved:
                    improved = True
                    n_moves += 1
        if improved:
            k = 0
        else:
            k += 1
    refresh_score(I, RI, RF, SC)
    return n_moves

### 13.6 · Repair: putting customers back

`best_insertion` scans a route for the cheapest feasible position and stops early once the customer can no longer be reached in time. `repair_regret` implements greedy (regret-1), regret-2 and regret-3 insertion with a per-(customer, route) cost cache; `repair_blink` is the SISR greedy insertion with random ordering rules and 1 % skipped positions. Customers that fit nowhere stay unassigned and are heavily penalised; such a state is never returned as a solution.

In [ ]:
%%alns_source kernels 6
# ======================================================== insertion =======
@njit
def best_insertion(dist, tt, ND, cap, I, F, RI, RF, u, r, blink):
    """Cheapest feasible position of unassigned u in nonempty route r.

    Returns (delta_cost, predecessor) or (INF, -1).  Each position is skipped
    with probability ``blink`` (SISR "blinks")."""
    if RF[LOAD, r] + ND[DEM, u] > cap + EPS_LOAD:
        return INF, -1
    best = INF
    bp = -1
    p = 0
    x = RI[FIRST, r]
    late_u = ND[TB, u] + EPS_TW
    while True:
        # Earliest service starts grow along the route; once u can no longer be
        # reached in time from p it cannot be from later nodes (metric travel).
        if p != 0 and F[ES, p] + ND[SVC, p] + tt[p, u] > late_u:
            break
        delta = dist[p, u] + dist[u, x] - dist[p, x]
        if delta < best:
            if blink <= 0.0 or np.random.random() >= blink:
                if insert_tw(tt, ND, F, p, u, x) <= EPS_TW:
                    best = delta
                    bp = p
        if x == 0:
            break
        p = x
        x = I[NXT, x]
    return best, bp


@njit
def repair_regret(inst, sol, pool, m, k, blink, insert_all):
    """Regret-k insertion (k == 1: parallel cheapest insertion).

    Customers pool[0..m-1] are inserted one by one; at each step the customer
    with the largest regret  sum_{h=2..k} (c_h - c_1)  over its k best routes is
    inserted at its cheapest position (ties: lower c_1).  Insertion costs are
    cached per (customer, route) and invalidated only for the modified route.
    Candidate routes are those of the customer's granular neighbours plus one
    unused route; if none admits the customer every route is scanned.

    Returns the number of customers left unassigned (moved to pool[0..])."""
    dist, tt, ND, cap, nbrs, empty_cost = inst[0], inst[1], inst[2], inst[3], inst[4], inst[8]
    I, F, RI, RF, CTR = sol[0], sol[1], sol[2], sol[3], sol[6]
    R = RI.shape[1]
    G = nbrs.shape[1]
    cost = np.empty((m, R))
    prv = np.empty((m, R), np.int64)
    valid = np.zeros((m, R), np.bool_)
    alive = np.ones(m, np.bool_)
    mark = np.zeros(R, np.int64)
    stamp = 0
    used = used_routes(RI)
    remaining = m
    while remaining > 0:
        best_i = -1
        best_score = -INF
        best_c1 = INF
        best_r = -1
        best_p = -1
        for i in range(m):
            if not alive[i]:
                continue
            u = pool[i]
            c1 = INF
            c2 = INF
            c3 = INF
            r1 = -1
            p1 = -1
            stamp += 1
            scan_all = insert_all == 1
            for phase in range(2):
                if phase == 1 and not scan_all:
                    break
                for jj in range(R if phase == 1 else G):
                    if phase == 0:
                        if insert_all == 1:
                            break
                        r = I[RTE, nbrs[u, jj]]
                    else:
                        r = jj
                    if r < 0 or mark[r] == stamp or RI[LEN, r] == 0:
                        continue
                    mark[r] = stamp
                    if not valid[i, r]:
                        c, p = best_insertion(dist, tt, ND, cap, I, F, RI, RF, u, r, blink)
                        cost[i, r] = c
                        prv[i, r] = p
                        valid[i, r] = True
                    c = cost[i, r]
                    if c < c1:
                        c3 = c2
                        c2 = c1
                        c1 = c
                        r1 = r
                        p1 = prv[i, r]
                    elif c < c2:
                        c3 = c2
                        c2 = c
                    elif c < c3:
                        c3 = c
                if phase == 0 and c1 >= INF:
                    scan_all = True
            if used < R and empty_cost[u] < INF:
                c = empty_cost[u]
                if c < c1:
                    c3 = c2
                    c2 = c1
                    c1 = c
                    r1 = -2
                    p1 = 0
                elif c < c2:
                    c3 = c2
                    c2 = c
                elif c < c3:
                    c3 = c
            if c1 >= INF:
                # no feasible position anywhere: inserting others only tightens
                # routes, so u stays unassigned for this repair.
                alive[i] = False
                remaining -= 1
                continue
            if k <= 1:
                score = -c1
            else:
                score = (c2 - c1) if c2 < INF else BIG_REGRET
                if k >= 3:
                    score += (c3 - c1) if c3 < INF else BIG_REGRET
            if score > best_score or (score == best_score and c1 < best_c1):
                best_score = score
                best_c1 = c1
                best_i = i
                best_r = r1
                best_p = p1
        if best_i < 0:
            break
        u = pool[best_i]
        if best_r == -2:
            best_r = empty_route(RI)
            used += 1
        link_after(I, RI, u, best_p, best_r)
        update_route(dist, tt, ND, I, F, RI, RF, CTR, best_r)
        alive[best_i] = False
        remaining -= 1
        for i in range(m):
            valid[i, best_r] = False
    left = 0
    for i in range(m):
        if I[RTE, pool[i]] < 0:
            pool[left] = pool[i]
            left += 1
    return left


@njit
def repair_blink(inst, sol, pool, m, blink, insert_all):
    """SISR-style greedy insertion with blinks in a randomly chosen order
    (random / largest demand / farthest from depot / closest / tightest window)."""
    dist, tt, ND, cap, nbrs, empty_cost = inst[0], inst[1], inst[2], inst[3], inst[4], inst[8]
    I, F, RI, RF, CTR = sol[0], sol[1], sol[2], sol[3], sol[6]
    R = RI.shape[1]
    G = nbrs.shape[1]
    key = np.empty(m)
    rule = np.random.random()
    for i in range(m):
        u = pool[i]
        if rule < 0.4:
            key[i] = np.random.random()
        elif rule < 0.7:
            key[i] = -ND[DEM, u]
        elif rule < 0.85:
            key[i] = -dist[0, u]
        elif rule < 0.9:
            key[i] = dist[0, u]
        else:
            key[i] = ND[TB, u] - ND[TA, u]
    order = np.argsort(key)
    tmp = pool[:m].copy()
    for i in range(m):
        pool[i] = tmp[order[i]]
    mark = np.zeros(R, np.int64)
    used = used_routes(RI)
    for i in range(m):
        u = pool[i]
        best = INF
        br = -1
        bp = -1
        scan_all = insert_all == 1
        for phase in range(2):
            if phase == 1 and not scan_all:
                break
            for jj in range(R if phase == 1 else G):
                if phase == 0:
                    if insert_all == 1:
                        break
                    r = I[RTE, nbrs[u, jj]]
                else:
                    r = jj
                if r < 0 or mark[r] == i + 1 or RI[LEN, r] == 0:
                    continue
                mark[r] = i + 1
                c, p = best_insertion(dist, tt, ND, cap, I, F, RI, RF, u, r, blink)
                if c < best:
                    best = c
                    br = r
                    bp = p
            if phase == 0 and best >= INF:
                scan_all = True
        if used < R and empty_cost[u] < best:
            best = empty_cost[u]
            br = -2
            bp = 0
        if best >= INF:
            continue
        if br == -2:
            br = empty_route(RI)
            used += 1
        link_after(I, RI, u, bp, br)
        update_route(dist, tt, ND, I, F, RI, RF, CTR, br)
    left = 0
    for i in range(m):
        if I[RTE, pool[i]] < 0:
            pool[left] = pool[i]
            left += 1
    return left

### 13.7 · Destroy: removing customers

Random removal; worst removal (disabled after tuning); Shaw removal of related customers (distance, windows, demand); SISR string removal of short strings from neighbouring routes (Christiaens & Vanden Berghe, 2020); and route removal, which empties a whole route and lets the fleet shrink.

In [ ]:
%%alns_source kernels 7
# ========================================================== destroy =======
@njit
def remove_customer(I, RI, u, pool, m, dirty):
    dirty[I[RTE, u]] = True
    unlink(I, RI, u)
    pool[m] = u
    return m + 1


@njit
def random_assigned(I):
    N = I.shape[1]
    for _ in range(64):
        u = np.random.randint(1, N)
        if I[RTE, u] >= 0:
            return u
    for u in range(1, N):
        if I[RTE, u] >= 0:
            return u
    return -1


@njit
def destroy_random(I, RI, pool, m, q, dirty):
    N = I.shape[1]
    cand = np.random.permutation(N - 1) + 1
    removed = 0
    for idx in range(N - 1):
        if removed >= q:
            break
        u = cand[idx]
        if I[RTE, u] >= 0:
            m = remove_customer(I, RI, u, pool, m, dirty)
            removed += 1
    return m


@njit
def destroy_worst(dist, I, RI, pool, m, q, dirty, p_worst):
    """Ropke & Pisinger worst removal: rank by distance saving, pick rank y^p."""
    N = I.shape[1]
    nodes = np.empty(N - 1, np.int64)
    gain = np.empty(N - 1)
    k = 0
    for u in range(1, N):
        if I[RTE, u] >= 0:
            p = I[PRV, u]
            x = I[NXT, u]
            nodes[k] = u
            gain[k] = -(dist[p, u] + dist[u, x] - dist[p, x])
            k += 1
    order = np.argsort(gain[:k])
    taken = np.zeros(k, np.bool_)
    left = k
    removed = 0
    while removed < q and left > 0:
        idx = int(np.random.random() ** p_worst * left)
        c = -1
        for j in range(k):
            if not taken[j]:
                c += 1
                if c == idx:
                    taken[j] = True
                    m = remove_customer(I, RI, nodes[order[j]], pool, m, dirty)
                    break
        left -= 1
        removed += 1
    return m


@njit
def destroy_shaw(shaw, I, RI, pool, m, q, dirty, p_shaw):
    """Shaw (relatedness) removal using precomputed relatedness lists."""
    L = shaw.shape[1]
    start = m
    seed = random_assigned(I)
    if seed < 0:
        return m
    m = remove_customer(I, RI, seed, pool, m, dirty)
    removed = 1
    cand = np.empty(max(L, 1), np.int64)
    while removed < q:
        r = pool[start + np.random.randint(0, m - start)]
        c = 0
        for j in range(L):
            v = shaw[r, j]
            if I[RTE, v] >= 0:
                cand[c] = v
                c += 1
        if c == 0:
            v = random_assigned(I)
            if v < 0:
                break
        else:
            v = cand[int(np.random.random() ** p_shaw * c)]
        m = remove_customer(I, RI, v, pool, m, dirty)
        removed += 1
    return m


@njit
def destroy_string(near, I, RI, pool, m, dirty, cbar, lmax, split_prob, beta, buf):
    """SISR string removal (Christiaens & Vanden Berghe, 2020): remove strings
    (optionally split around a preserved substring) from routes adjacent to a
    random seed customer.  Its size is governed by cbar/lmax, not by q."""
    R = RI.shape[1]
    nonempty = 0
    assigned = 0
    for r in range(R):
        if RI[LEN, r] > 0:
            nonempty += 1
            assigned += RI[LEN, r]
    if nonempty == 0:
        return m
    ls_max = min(float(lmax), assigned / nonempty)
    ks_max = 4.0 * cbar / (1.0 + ls_max) - 1.0
    ks = int(np.random.random() * ks_max) + 1
    seed = random_assigned(I)
    ruined = np.zeros(R, np.bool_)
    n_ruined = 0
    for j in range(-1, near.shape[1]):
        if n_ruined >= ks:
            break
        c = seed if j < 0 else near[seed, j]
        r = I[RTE, c]
        if r < 0 or ruined[r]:
            continue
        ruined[r] = True
        n_ruined += 1
        t = collect(I, RI, r, buf)
        lt_max = min(float(t), ls_max)
        l = int(np.random.random() * lt_max) + 1
        pc = I[POS, c]
        keep = 0
        if np.random.random() < split_prob and l < t:
            keep = 1
            while l + keep < t and np.random.random() > beta:
                keep += 1
        total = l + keep
        lo = max(0, pc - total + 1)
        hi = min(pc, t - total)
        s = lo + np.random.randint(0, hi - lo + 1)
        keep_start = s + np.random.randint(0, l + 1) if keep > 0 else -1
        for k in range(s, s + total):
            if keep > 0 and keep_start <= k < keep_start + keep:
                continue
            m = remove_customer(I, RI, buf[k], pool, m, dirty)
    return m


@njit
def destroy_route(I, RI, pool, m, dirty):
    """Remove all customers of one random route (route-count reduction)."""
    R = RI.shape[1]
    nonempty = 0
    for r in range(R):
        if RI[LEN, r] > 0:
            nonempty += 1
    if nonempty == 0:
        return m
    pick = np.random.randint(0, nonempty)
    for r in range(R):
        if RI[LEN, r] > 0:
            if pick == 0:
                u = RI[FIRST, r]
                while u != 0:
                    x = I[NXT, u]
                    m = remove_customer(I, RI, u, pool, m, dirty)
                    u = x
                break
            pick -= 1
    return m

### 13.8 · The ALNS iteration

Copy the current solution → pick a destroy and a repair operator by roulette on their adaptive weights → destroy $q\in[5,\min(40,0.2n)]$ customers → repair → VND → accept with simulated annealing. Operators earn 33 / 9 / 13 points for a new best / an improvement / an accepted worse solution; weights are updated every 100 iterations with reaction factor 0.1. The loop runs in Numba in chunks of iterations; Python checks the clock between chunks.

In [ ]:
%%alns_source kernels 8
# ===================================================== ALNS main loop ======
@njit
def roulette(w):
    total = 0.0
    for i in range(w.shape[0]):
        total += w[i]
    x = np.random.random() * total
    for i in range(w.shape[0]):
        x -= w[i]
        if x <= 0.0 and w[i] > 0.0:
            return i
    for i in range(w.shape[0] - 1, -1, -1):
        if w[i] > 0.0:
            return i
    return 0


@njit
def construct(inst, sol, pool, order, G):
    """Initial solution: parallel cheapest insertion of every customer
    (regret-1, granular candidate routes), followed by VND."""
    I, RI, RF, SC = sol[0], sol[2], sol[3], sol[5]
    N = I.shape[1]
    m = 0
    for u in range(1, N):
        if I[RTE, u] < 0:
            pool[m] = u
            m += 1
    repair_regret(inst, sol, pool, m, 1, 0.0, 0)
    refresh_score(I, RI, RF, SC)
    buf = np.empty(N, np.int64)
    if SC[1] == 0:
        vnd(inst, sol, order, G, buf)
    refresh_score(I, RI, RF, SC)


@njit
def alns_chunk(inst, cur, cand, best, IP, P, wd, wr, sd, sr, ud, ur,
               order, n_iter, temperature, stats, pool, buf):
    """Run ``n_iter`` ALNS iterations (destroy, repair, VND, SA acceptance)."""
    dist, tt, ND, near, shaw = inst[0], inst[1], inst[2], inst[5], inst[6]
    cI, cF, cRI, cRF, cCTR = cand[0], cand[1], cand[2], cand[3], cand[6]
    N = cI.shape[1]
    R = cRI.shape[1]
    penalty = P[P_PENALTY]
    G = IP[IP_VND_NEIGHBOURS]
    dirty = np.zeros(R, np.bool_)
    for _ in range(n_iter):
        IP[IP_ITER] += 1
        it = IP[IP_ITER]
        copy_sol(cur, cand)
        d = roulette(wd)
        rp = roulette(wr)
        q = IP[IP_QMIN]
        if IP[IP_QMAX] > IP[IP_QMIN]:
            q = np.random.randint(IP[IP_QMIN], IP[IP_QMAX] + 1)
        # customers still unassigned stay in the pool
        m = 0
        for u in range(1, N):
            if cI[RTE, u] < 0:
                pool[m] = u
                m += 1
        dirty[:] = False
        if d == D_RANDOM:
            m = destroy_random(cI, cRI, pool, m, q, dirty)
        elif d == D_WORST:
            m = destroy_worst(dist, cI, cRI, pool, m, q, dirty, P[P_WORST_P])
        elif d == D_SHAW:
            m = destroy_shaw(shaw, cI, cRI, pool, m, q, dirty, P[P_SHAW_P])
        elif d == D_STRING:
            m = destroy_string(near, cI, cRI, pool, m, dirty, P[P_SISR_CBAR],
                               IP[IP_SISR_LMAX], P[P_SISR_SPLIT], P[P_SISR_BETA], buf)
        else:
            m = destroy_route(cI, cRI, pool, m, dirty)
        for r in range(R):
            if dirty[r]:
                update_route(dist, tt, ND, cI, cF, cRI, cRF, cCTR, r)
        if rp == R_GREEDY:
            left = repair_regret(inst, cand, pool, m, 1, 0.0, IP[IP_INSERT_ALL])
        elif rp == R_REGRET2:
            left = repair_regret(inst, cand, pool, m, 2, 0.0, IP[IP_INSERT_ALL])
        elif rp == R_REGRET3:
            left = repair_regret(inst, cand, pool, m, 3, 0.0, IP[IP_INSERT_ALL])
        else:
            left = repair_blink(inst, cand, pool, m, P[P_BLINK], IP[IP_INSERT_ALL])
        refresh_score(cI, cRI, cRF, cand[5])
        if left == 0:
            stats[ST_VND_MOVES] += vnd(inst, cand, order, G, buf)
        else:
            stats[ST_INFEASIBLE_CAND] += 1
        f_cand = objective(cand[5], penalty)
        f_cur = objective(cur[5], penalty)
        f_best = objective(best[5], penalty)
        score = 0.0
        if f_cand < f_best - EPS_COST:
            copy_sol(cand, best)
            copy_sol(cand, cur)
            score = P[P_SIG1]
            IP[IP_LAST_IMPROVE] = it
            stats[ST_NEW_BEST] += 1
            stats[ST_ACCEPT] += 1
        elif f_cand < f_cur - EPS_COST:
            copy_sol(cand, cur)
            score = P[P_SIG2]
            stats[ST_IMPROVE_CUR] += 1
            stats[ST_ACCEPT] += 1
        elif f_cand <= f_cur + EPS_COST:
            copy_sol(cand, cur)          # sideways move: accept, no reward
            stats[ST_ACCEPT] += 1
        elif temperature > 0.0:
            if np.random.random() < np.exp(-(f_cand - f_cur) / temperature):
                copy_sol(cand, cur)
                score = P[P_SIG3]
                stats[ST_ACCEPT] += 1
        sd[d] += score
        ud[d] += 1.0
        sr[rp] += score
        ur[rp] += 1.0
        stats[ST_ITERS] += 1
        if IP[IP_SEGMENT] > 0 and it % IP[IP_SEGMENT] == 0:
            rho = P[P_REACTION]
            for j in range(wd.shape[0]):
                if wd[j] > 0.0 and ud[j] > 0.0:
                    wd[j] = max(1e-3, wd[j] * (1.0 - rho) + rho * sd[j] / ud[j])
                sd[j] = 0.0
                ud[j] = 0.0
            for j in range(wr.shape[0]):
                if wr[j] > 0.0 and ur[j] > 0.0:
                    wr[j] = max(1e-3, wr[j] * (1.0 - rho) + rho * sr[j] / ur[j])
                sr[j] = 0.0
                ur[j] = 0.0
        if IP[IP_RESTART] > 0 and it - IP[IP_LAST_IMPROVE] >= IP[IP_RESTART]:
            copy_sol(best, cur)
            IP[IP_LAST_IMPROVE] = it
            stats[ST_RESTARTS] += 1

## 14 · The driver (`solver`)

`solve(instance, config, seed, time_limit)` follows the shared `Solver` protocol of `scripts/benchmark.py`: it receives only the `CVRPTWInstance` — never reference routes or costs — and returns customer-only routes in internal indices 1..n with a claimed cost that the independent evaluator re-checks.

### 14.1 · Provenance and the time budget's origin

In [ ]:
%%alns_source solver 1
"""ALNS + VND driver: preprocessing, time management, trace and result packaging.

Public entry point: ``solve(instance, config, seed, time_limit) -> SolverResult``,
matching the ``scripts.benchmark.Solver`` protocol.  The solver receives only the
instance (never reference routes or costs), returns customer-only routes in
internal indices 1..n, and lets the shared evaluator re-check everything.
"""
from __future__ import annotations

import hashlib
import os
from pathlib import Path
import time

import numpy as np

from scripts.benchmark import SolverResult
from scripts.cvrptw_model import CVRPTWInstance, require_cvrptw

from . import kernels as K
from ._jit import JIT_ENABLED
from .config import ALNSConfig, DESTROY_OPERATORS, NEIGHBOURHOODS, REPAIR_OPERATORS

VERSION = "alns-vnd-1.0"
_SOURCES = ("__init__.py", "_jit.py", "config.py", "kernels.py", "solver.py")


# Set by the main.ipynb loader when the solver was compiled from notebook cells
# that differ from these files: {module: sha256 of the notebook source}.
NOTEBOOK_SOURCE_SHA256 = None


def source_fingerprint():
    """SHA256 of the solver sources (recorded with every run)."""
    if NOTEBOOK_SOURCE_SHA256 is not None:
        return {"origin": "main.ipynb", **NOTEBOOK_SOURCE_SHA256}
    root = Path(__file__).resolve().parent
    return {name: hashlib.sha256((root / name).read_bytes()).hexdigest() for name in _SOURCES}


def _process_start():
    """Monotonic-clock estimate of when this process started (or None)."""
    try:
        import psutil
        age = time.time() - psutil.Process(os.getpid()).create_time()
        return time.perf_counter() - max(0.0, age)
    except Exception:
        return None

### 14.2 · Preprocessing

Distance, travel-time and window arrays; granular neighbour lists by time-window-aware proximity; nearest-neighbour lists for string removal; relatedness lists for Shaw removal; and the cost of serving each customer alone.

In [ ]:
%%alns_source solver 2
# ----------------------------------------------------------- preprocessing --
def _top_k(score, k):
    """Indices of the k smallest entries of each row (row-wise, sorted)."""
    if k <= 0:
        return np.zeros((score.shape[0], 0), np.int64)
    part = np.argpartition(score, k - 1, axis=1)[:, :k]
    rows = np.arange(score.shape[0])[:, None]
    order = np.argsort(score[rows, part], axis=1, kind="stable")
    return np.ascontiguousarray(part[rows, order], dtype=np.int64)


def build_instance_data(instance: CVRPTWInstance, cfg: ALNSConfig):
    """Numba-ready tuple with distances, windows and neighbour lists."""
    dist = np.ascontiguousarray(instance.distances, dtype=np.float64)
    tt = np.ascontiguousarray(instance.travel_times, dtype=np.float64)
    ta = np.ascontiguousarray(instance.time_windows[:, 0], dtype=np.float64)
    tb = np.ascontiguousarray(instance.time_windows[:, 1], dtype=np.float64)
    svc = np.ascontiguousarray(instance.service_durations, dtype=np.float64)
    dem = np.ascontiguousarray(instance.demands, dtype=np.float64)
    N = dem.shape[0]
    n = N - 1
    cap = float(instance.capacity)
    sym = int(np.array_equal(dist, dist.T) and np.array_equal(tt, tt.T))

    # Depot -> u -> depot (earliest schedule); INF when u cannot be served alone.
    arrive = np.maximum(ta[0] + tt[0, :], ta)
    alone_ok = (arrive <= tb + K.EPS_TW) & (arrive + svc + tt[:, 0] <= tb[0] + K.EPS_TW)
    empty_cost = np.where(alone_ok, dist[0, :] + dist[:, 0], K.INF)
    empty_cost[0] = K.INF

    c = slice(1, N)
    # Granular proximity (Vidal et al. 2013): distance plus penalties for the
    # waiting / lateness incurred if j directly follows i.
    late = np.maximum(ta[c, None] + svc[c, None] + tt[c, c] - tb[None, c], 0.0)
    wait = np.maximum(ta[None, c] - svc[c, None] - tt[c, c] - tb[c, None], 0.0)
    prox = dist[c, c] + cfg.tw_wait_weight * wait + cfg.tw_warp_weight * late
    prox = np.minimum(prox, prox.T)
    np.fill_diagonal(prox, np.inf)
    G = min(cfg.n_neighbours, n - 1)
    nbrs = np.zeros((N, G), np.int64)
    nbrs[1:] = _top_k(prox, G) + 1

    d_sym = np.minimum(dist[c, c], dist[c, c].T).copy()
    np.fill_diagonal(d_sym, np.inf)
    near = np.zeros((N, min(cfg.near_list, n - 1)), np.int64)
    near[1:] = _top_k(d_sym, near.shape[1]) + 1

    w_d, w_t, w_q = cfg.shaw_weights
    dmax = max(float(np.max(dist[c, c])), 1e-12)
    tspan = max(float(tb[c].max() - ta[c].min()), 1e-12)
    qmax = max(float(dem[c].max()), 1e-12)
    related = (w_d * d_sym / dmax
               + w_t * (np.abs(ta[c, None] - ta[None, c]) + np.abs(tb[c, None] - tb[None, c])) / (2 * tspan)
               + w_q * np.abs(dem[c, None] - dem[None, c]) / qmax)
    np.fill_diagonal(related, np.inf)
    shaw = np.zeros((N, min(cfg.shaw_list, n - 1)), np.int64)
    shaw[1:] = _top_k(related, shaw.shape[1]) + 1

    ND = np.ascontiguousarray(np.vstack([ta, tb, svc, dem]), dtype=np.float64)
    return (dist, tt, ND, cap, nbrs, near, shaw, sym,
            np.ascontiguousarray(empty_cost, dtype=np.float64))


def new_solution(N, R, ctr):
    I = np.zeros((4, N), np.int64)
    I[K.RTE, :] = -1
    I[K.RTE, 0] = -2  # depot is never a customer
    F = np.zeros((10, N), np.float64)
    RI = np.zeros((4, R), np.int64)
    RF = np.zeros((2, R), np.float64)
    TST = np.zeros((K.N_NEIGHBOURHOODS, N), np.int64)
    SC = np.zeros(2, np.float64)
    SC[1] = N - 1
    return (I, F, RI, RF, TST, SC, ctr)


def decode_routes(sol):
    I, RI = sol[0], sol[2]
    routes = []
    for r in range(RI.shape[1]):
        if RI[K.LEN, r] == 0:
            continue
        route = []
        u = int(RI[K.FIRST, r])
        while u != 0:
            route.append(u)
            u = int(I[K.NXT, u])
        routes.append(route)
    return routes


def route_cost(instance, routes):
    """Claimed cost with the instance's own scoring (DIMACS integer tenths when cached)."""
    ticks = instance.distance_tenths
    if ticks is not None:
        total = 0
        for r in routes:
            path = [0] + r + [0]
            total += int(sum(int(ticks[i, j]) for i, j in zip(path, path[1:])))
        return total / 10
    return float(sum(instance.distances[i, j] for r in routes for i, j in zip([0] + r, r + [0])))

### 14.3 · `solve`: construction, time-managed search, result

Parallel cheapest insertion + VND builds the first solution. The temperature falls exponentially with the elapsed fraction of the budget; the search stops at `time_fraction × limit − time_reserve`. The best solution is returned with a trace (best cost over time) and complete metadata: full configuration, iterations, operator weights, time to best and source hashes.

In [ ]:
%%alns_source solver 3
# ------------------------------------------------------------------- solve --
def solve(instance: CVRPTWInstance, config: dict | None, seed: int, time_limit: float) -> SolverResult:
    """Run ALNS + VND under a wall-clock limit (seconds) and return the best
    feasible solution found.  ``config`` overrides ALNSConfig defaults."""
    call_start = time.perf_counter()
    require_cvrptw(instance)
    cfg = ALNSConfig.from_dict(config)
    if not np.isfinite(time_limit) or time_limit <= 0:
        raise ValueError("time_limit must be positive and finite")
    origin = _process_start()
    if cfg.clock == "process":
        if origin is None:
            raise RuntimeError("clock='process' requires psutil to read the process start time")
        budget_origin = min(origin, call_start)
    else:
        budget_origin = call_start
    deadline = budget_origin + time_limit * cfg.time_fraction - cfg.time_reserve
    if deadline <= call_start:
        deadline = call_start + 0.5 * time_limit * cfg.time_fraction

    seed = int(seed)
    K.seed_rng(seed % (2 ** 32))
    inst = build_instance_data(instance, cfg)
    N = inst[2].shape[1]
    n = N - 1
    R = int(min(instance.max_vehicles, n))
    ctr = np.zeros(1, np.int64)
    cur = new_solution(N, R, ctr)
    cand = new_solution(N, R, ctr)
    best = new_solution(N, R, ctr)
    order = np.array([NEIGHBOURHOODS.index(nb) for nb in cfg.neighbourhoods], np.int64)
    pool = np.zeros(N, np.int64)
    buf = np.zeros(N, np.int64)

    K.construct(inst, cur, pool, order, cfg.vnd_neighbours)
    K.copy_sol(cur, best)
    construct_seconds = time.perf_counter() - call_start
    initial_cost = float(cur[5][0])
    initial_unassigned = int(cur[5][1])

    max_dist = float(np.max(inst[0])) if n else 0.0
    penalty = 1e3 * (1.0 + 2.0 * max_dist)
    q_max = int(max(1, min(cfg.destroy_max, round(cfg.destroy_max_fraction * n), n)))
    q_min = int(min(cfg.destroy_min, q_max))
    IP = np.zeros(K.N_IP, np.int64)
    IP[K.IP_QMIN], IP[K.IP_QMAX] = q_min, q_max
    IP[K.IP_SEGMENT], IP[K.IP_RESTART] = cfg.segment, cfg.restart_after
    IP[K.IP_SISR_LMAX] = cfg.sisr_max_string
    IP[K.IP_INSERT_ALL] = int(cfg.insert_all_routes)
    IP[K.IP_VND_NEIGHBOURS] = cfg.vnd_neighbours
    P = np.zeros(K.N_P, np.float64)
    P[K.P_WORST_P], P[K.P_SHAW_P] = cfg.worst_randomness, cfg.shaw_randomness
    P[K.P_SISR_CBAR], P[K.P_SISR_SPLIT], P[K.P_SISR_BETA] = cfg.sisr_avg_removed, cfg.sisr_split_rate, cfg.sisr_split_depth
    P[K.P_BLINK] = cfg.blink_rate
    P[K.P_SIG1], P[K.P_SIG2], P[K.P_SIG3] = cfg.score_new_best, cfg.score_better, cfg.score_accepted
    P[K.P_REACTION], P[K.P_PENALTY] = cfg.reaction, penalty
    wd = np.array(cfg.destroy_weights, np.float64)
    wr = np.array(cfg.repair_weights, np.float64)
    sd, ud = np.zeros_like(wd), np.zeros_like(wd)
    sr, ur = np.zeros_like(wr), np.zeros_like(wr)
    stats = np.zeros(K.N_ST, np.float64)

    used = max(1, K.used_routes(cur[2]))
    mean_arc = initial_cost / (n + used) if n else 1.0
    t_start = cfg.temp_start * mean_arc
    t_end = cfg.temp_end * mean_arc

    def best_cost():
        return float(best[5][0]) if best[5][1] == 0 else None

    trace = [dict(seconds=round(time.perf_counter() - budget_origin, 4), iteration=0,
                  best_cost=best_cost(), current_cost=float(cur[5][0]), event="construction")]
    chunk = 1
    stop_reason = "time_limit"
    last_best = objective = K.objective(best[5], penalty)
    time_to_best = time.perf_counter() - budget_origin
    last_trace = time.perf_counter()
    search_start = time.perf_counter()
    while True:
        now = time.perf_counter()
        if now >= deadline:
            break
        if cfg.max_iterations and IP[K.IP_ITER] >= cfg.max_iterations:
            stop_reason = "max_iterations"
            break
        span = max(deadline - search_start, 1e-9)
        frac = min(1.0, (now - search_start) / span)
        if cfg.max_iterations:
            frac = max(frac, IP[K.IP_ITER] / cfg.max_iterations)
        temperature = t_start * (t_end / t_start) ** frac if t_start > 0 else 0.0
        n_iter = chunk
        if cfg.max_iterations:
            n_iter = int(min(n_iter, cfg.max_iterations - IP[K.IP_ITER]))
        t0 = time.perf_counter()
        K.alns_chunk(inst, cur, cand, best, IP, P, wd, wr, sd, sr, ud, ur,
                     order, n_iter, temperature, stats, pool, buf)
        elapsed = time.perf_counter() - t0
        # adapt the chunk so that time checks happen every ~chunk_seconds
        if elapsed < cfg.chunk_seconds / 2:
            chunk = min(chunk * 2, 1 << 20)
        elif elapsed > cfg.chunk_seconds * 2 and chunk > 1:
            chunk = max(1, chunk // 2)
        remaining = deadline - time.perf_counter()
        per_iter = elapsed / max(n_iter, 1)
        if per_iter > 0:
            chunk = max(1, min(chunk, int(remaining / per_iter / 2) + 1))
        objective = K.objective(best[5], penalty)
        stamp = time.perf_counter()
        if objective < last_best - K.EPS_COST:
            last_best = objective
            time_to_best = stamp - budget_origin
            trace.append(dict(seconds=round(stamp - budget_origin, 4), iteration=int(IP[K.IP_ITER]),
                              best_cost=best_cost(), current_cost=float(cur[5][0]),
                              temperature=temperature, event="improvement"))
            last_trace = stamp
        elif stamp - last_trace > 2.0:
            trace.append(dict(seconds=round(stamp - budget_origin, 4), iteration=int(IP[K.IP_ITER]),
                              best_cost=best_cost(), current_cost=float(cur[5][0]),
                              temperature=temperature, event="heartbeat"))
            last_trace = stamp

    routes = decode_routes(best)
    feasible_state = int(best[5][1]) == 0
    claimed = route_cost(instance, routes) if feasible_state else None
    finish = time.perf_counter()
    trace.append(dict(seconds=round(finish - budget_origin, 4), iteration=int(IP[K.IP_ITER]),
                      best_cost=claimed, current_cost=float(cur[5][0]), event="stop"))
    metadata = dict(
        algorithm="ALNS+VND", version=VERSION, seed=seed, config=cfg.to_dict(),
        jit=JIT_ENABLED, numba_version=_numba_version(), source_sha256=source_fingerprint(),
        stop_reason=stop_reason, iterations=int(IP[K.IP_ITER]),
        construction_seconds=construct_seconds,
        search_seconds=finish - search_start, solver_call_seconds=finish - call_start,
        process_start_known=origin is not None,
        pre_call_seconds=(call_start - origin) if origin is not None else None,
        time_to_best_seconds=time_to_best, deadline_seconds=deadline - budget_origin,
        initial_cost=initial_cost if initial_unassigned == 0 else None, initial_unassigned=initial_unassigned,
        unassigned=int(best[5][1]), routes_used=len(routes), max_routes=R,
        destroy_size=[q_min, q_max], neighbours=int(inst[4].shape[1]),
        final_destroy_weights=dict(zip(DESTROY_OPERATORS, wd.tolist())),
        final_repair_weights=dict(zip(REPAIR_OPERATORS, wr.tolist())),
        stats=dict(iterations=int(stats[K.ST_ITERS]), accepted=int(stats[K.ST_ACCEPT]),
                   improved_current=int(stats[K.ST_IMPROVE_CUR]), new_best=int(stats[K.ST_NEW_BEST]),
                   restarts=int(stats[K.ST_RESTARTS]), incomplete_candidates=int(stats[K.ST_INFEASIBLE_CAND]),
                   vnd_moves=int(stats[K.ST_VND_MOVES])),
        temperature=dict(start=t_start, end=t_end, mean_arc=mean_arc),
    )
    if not feasible_state:
        return SolverResult(None, status="no_solution", trace=trace, metadata=metadata)
    return SolverResult(routes, departure_times=None, status="completed", claimed_cost=claimed,
                        trace=trace, metadata=metadata)

### 14.4 · Warm-up

In [ ]:
%%alns_source solver 4
def _numba_version():
    try:
        import numba
        return numba.__version__
    except ImportError:
        return None


def warmup():
    """Compile (or load cached) kernels on a tiny instance."""
    xy = np.array([[0, 0], [1, 0], [2, 0], [0, 2], [3, 3], [1, 1]], float)
    d = np.floor(10 * np.hypot(*(xy[:, None, :] - xy[None, :, :]).transpose(2, 0, 1))) / 10
    inst = CVRPTWInstance("warmup", [0, 1, 1, 1, 1, 1], 3, 4, d, [[0, 100]] * 6, [0, 1, 1, 1, 1, 1])
    return solve(inst, {"max_iterations": 50, "time_reserve": 0.0}, 0, 30.0)

## 15 · Running the solver

### 15.1 · Load the solver

This cell assembles the modules above. With unchanged cells it loads the precompiled kernels (about a second); after an edit, or with `ECOROUTE_ALNS_FROM_NOTEBOOK=1`, it compiles the cells themselves (30–60 s). The warm-up solves a 5-customer instance so that no compilation happens inside a timed run.

In [ ]:
ALNS_FROM_NOTEBOOK = os.environ.get("ECOROUTE_ALNS_FROM_NOTEBOOK", "0") == "1"
alns, ALNS_MODE = load_alns(force_notebook=ALNS_FROM_NOTEBOOK)
_started = time.perf_counter()
alns.warmup()  # loads cached machine code, or compiles the notebook version
print(f"ALNS + VND {alns.VERSION}: {ALNS_MODE}")
print(f"Numba JIT enabled: {alns.solver.JIT_ENABLED}; kernels ready in {time.perf_counter() - _started:.1f} s")

### 15.2 · Independent evaluation helpers

The solver's own feasibility and cost are never trusted: every result is re-checked by `check_cvrptw` (Section 10A) and re-scored with DIMACS integer tenths. The published BKS is itself re-validated before it is used as a gap denominator. Gaps are to **best-known** solutions, not proven optima.

In [ ]:
def reference_cost(vrp_path):
    """Published BKS of an instance, independently validated and re-scored."""
    instance = load_cvrptw(vrp_path)
    ref = vrplib.read_solution(str(Path(vrp_path).with_suffix(".sol")))
    audit = check_cvrptw(instance, [list(r) for r in ref["routes"]])
    if not audit["feasible"] or abs(audit["distance"] - float(ref["cost"])) > 1e-7:
        raise ValueError(f"Reference for {vrp_path} failed validation: {audit['errors']}")
    return audit["distance"]


def evaluate_alns(instance, result, bks):
    """Feasibility, independent DIMACS cost and gap of one solver result."""
    if result.routes is None:
        return dict(feasible=False, cost=None, gap_percent=None, vehicles=None,
                    cost_agrees=False, errors=["no routes returned"])
    audit = check_cvrptw(instance, result.routes)
    feasible = audit["feasible"]
    cost = audit["distance"]
    return dict(feasible=feasible, cost=cost,
                gap_percent=100 * (cost - bks) / bks if feasible else None,
                vehicles=len(audit["active_routes"]), errors=audit["errors"],
                cost_agrees=feasible and abs(cost - result.claimed_cost) <= 1e-7)

### 15.3 · Live demonstration

A short run on four **development** instances (used for tuning, never for the final test) of 100–400 customers. `ECOROUTE_DEMO_SECONDS` sets the per-instance limit (default 10 s, a third of the 0.3 s/customer benchmark budget, so expect slightly larger gaps than §15.4). The solver runs in this process with `clock="call"`; benchmark campaigns instead isolate every attempt in a worker with a hard kill. Any infeasible result stops the notebook.

In [ ]:
DEMO_SECONDS = float(os.environ.get("ECOROUTE_DEMO_SECONDS", "10"))
DEMO_INSTANCES = ["Solomon/R102", "Solomon/RC107", "GH200/RC2_2_8", "GH400/R1_4_10"]  # development split
DEMO_SEED = 0
demo_rows, demo_runs = [], {}
for name in DEMO_INSTANCES:
    vrp = DATA_ROOT / f"{name}.vrp"
    if not (vrp.is_file() and vrp.with_suffix(".sol").is_file()):
        if STRICT_DATA:
            raise FileNotFoundError(vrp)
        print(f"{name}: NOT RUN (instance or reference unavailable)")
        continue
    instance = load_cvrptw(vrp)
    bks = reference_cost(vrp)
    _started = time.perf_counter()
    result = alns.solve(instance, {}, DEMO_SEED, DEMO_SECONDS)
    wall = time.perf_counter() - _started
    check = evaluate_alns(instance, result, bks)
    if not (check["feasible"] and check["cost_agrees"]):
        raise AssertionError(f"{name}: solver output rejected by the validator: {check['errors']}")
    demo_runs[name] = (instance, result, bks)
    demo_rows.append(dict(instance=name, customers=instance.n_customers, K=instance.max_vehicles,
                          limit_s=DEMO_SECONDS, wall_s=round(wall, 1), iterations=result.metadata["iterations"],
                          routes=check["vehicles"], cost=check["cost"], BKS=bks,
                          gap_percent=round(check["gap_percent"], 3), feasible=check["feasible"]))
display(pd.DataFrame(demo_rows))

In [ ]:
if demo_runs:
    shown = "GH200/RC2_2_8" if "GH200/RC2_2_8" in demo_runs else next(iter(demo_runs))
    instance, result, bks = demo_runs[shown]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
    xy = instance.coordinates
    for k, route in enumerate(result.routes):
        path = [0] + route + [0]
        ax1.plot(xy[path, 0], xy[path, 1], "-", lw=1.2, color=plt.cm.tab20(k % 20))
    ax1.scatter(xy[1:, 0], xy[1:, 1], s=9, c="black", zorder=3)
    ax1.scatter(*xy[0], marker="s", s=90, c="red", zorder=4, label="depot")
    ax1.set(title=f"{shown}: {len(result.routes)} routes, cost {result.claimed_cost:.1f} (BKS {bks:.1f})",
            aspect="equal")
    ax1.legend(loc="upper right")
    for name, (inst, res, ref) in demo_runs.items():
        points = [(t["seconds"], 100 * (t["best_cost"] - ref) / ref) for t in res.trace
                  if t.get("best_cost") is not None]
        ax2.step(*zip(*points), where="post", label=name)
    ax2.set(xlabel="seconds since solve() was called", ylabel="best gap to BKS (%)",
            title="Convergence of the live runs")
    ax2.set_yscale("symlog", linthresh=0.5)
    ax2.legend()
    fig.tight_layout()
    plt.show()
else:
    print("Live demonstration plots: NOT RUN")

### 15.4 · Frozen experiments: tuning and the held-out test

The experiments themselves are run outside the notebook with `scripts/run_alns.py`: every attempt executes in an isolated worker under a hard wall-clock limit (start-up included) and is independently re-validated by `scripts/benchmark.py`. The cells below only **load the committed campaign files** in `results/alns-vnd/`; they do not recompute them.

* **Split** (`split.json`, seed 20261110, frozen before tuning): one test and one development instance per (size family × class), 284 untouched reserve instances.
* **Tuning**: 7 development campaigns (36 instances × 2 seeds, 0.1 s/customer) changed one setting at a time.
* **Test** (`test-001`): 36 held-out instances × 10 seeds, `time_limit = max(10, 0.3·n)` s, configuration frozen beforehand, Apple M2 Max with 6 concurrent single-threaded workers.

In [ ]:
ALNS_RESULTS = PROJECT_ROOT / "results" / "alns-vnd"
TEST_CAMPAIGN = ALNS_RESULTS / "test-001"
DEV_CAMPAIGNS = {"dev-001-baseline": "initial defaults", "dev-002-coolT": "SA 0.15 -> 0.005",
                 "dev-003-hotT": "SA 1.5 -> 0.02", "dev-004-hot2": "SA 4.0 -> 0.05",
                 "dev-005-hotT-g20": "+ 20 VND neighbours", "dev-006-q20": "+ destroy max 20",
                 "dev-007-noworst": "+ worst removal off (frozen)"}


def campaign_runs(directory):
    return pd.DataFrame([json.loads(line) for line in (directory / "runs.jsonl").read_text().splitlines()])


def instance_means(runs):
    ok = runs[runs.feasible & runs.gap_percent.notna()]
    return ok.groupby("instance_id").gap_percent.mean()


tuning = []
for campaign, change in DEV_CAMPAIGNS.items():
    if (ALNS_RESULTS / campaign / "runs.jsonl").is_file():
        runs = campaign_runs(ALNS_RESULTS / campaign)
        tuning.append(dict(campaign=campaign, change=change, attempts=len(runs), feasible=int(runs.feasible.sum()),
                           mean_gap=instance_means(runs).mean(), worst_run=runs.gap_percent.max()))
if tuning:
    print("Tuning history (development split only):")
    display(pd.DataFrame(tuning).round(3))
else:
    print("Tuning history: NOT RUN (results/alns-vnd/dev-* unavailable)")

if (TEST_CAMPAIGN / "runs.jsonl").is_file():
    test_runs = campaign_runs(TEST_CAMPAIGN)
    verdict = pd.read_csv(TEST_CAMPAIGN / "comparison.csv").iloc[0]
    means = instance_means(test_runs)
    gaps = test_runs.loc[test_runs.feasible, "gap_percent"]
    summary = {
        "attempts (planned 360)": len(test_runs),
        "independently feasible": int(test_runs.feasible.sum()),
        "within hard time limit": int((test_runs.budget_respected == True).sum()),  # noqa: E712
        "mean gap, equal weight per instance (%)": round(means.mean(), 3),
        "median instance mean gap (%)": round(means.median(), 3),
        "P90 run gap (%)": round(float(np.percentile(gaps, 90)), 3),
        "worst run gap (%)": round(gaps.max(), 3),
        "runs above 7 %": int((gaps > 7).sum()),
        "shared evaluator assessment": verdict.assessment,
    }
    display(pd.Series(summary, name="held-out test-001").to_frame())
    per_instance = pd.read_csv(TEST_CAMPAIGN / "alns_instances.csv")
    display(per_instance.groupby("family", sort=False).agg(
        customers=("n", "first"), limit_s=("time_limit", "first"), mean_gap=("gap_mean", "mean"),
        worst_run=("gap_worst", "max"), mean_seed_sd=("gap_sd", "mean")).round(3))
    display(per_instance.groupby("cls").agg(mean_gap=("gap_mean", "mean"), worst_run=("gap_worst", "max")).round(3))
else:
    test_runs = None
    print("Held-out test results: NOT RUN (results/alns-vnd/test-001 unavailable)")

In [ ]:
if test_runs is not None:
    per_instance = pd.read_csv(TEST_CAMPAIGN / "alns_instances.csv").sort_values(["n", "instance"])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [2, 1]})
    colours = {c: plt.cm.tab10(i) for i, c in enumerate(["C1", "C2", "R1", "R2", "RC1", "RC2"])}
    x = np.arange(len(per_instance))
    for cls, group in per_instance.groupby("cls"):
        idx = x[per_instance.cls.values == cls]
        ax1.errorbar(idx, group.gap_mean, yerr=[group.gap_mean - group.gap_best, group.gap_worst - group.gap_mean],
                     fmt="o", ms=4, capsize=2, color=colours[cls], label=cls)
    ax1.axhline(3, ls="--", color="green", lw=1, label="3 % mean target")
    ax1.axhline(7, ls=":", color="red", lw=1, label="7 % investigation")
    ticks = pd.Series(x).groupby(per_instance.n.values).mean()  # centre of each size group
    ax1.set_xticks(ticks.values, [f"n={n}" for n in ticks.index])
    ax1.set(ylabel="gap to published BKS (%)", title="Held-out test: mean gap per instance (bars: best-worst of 10 seeds)")
    ax1.legend(ncol=4, fontsize=8)
    if tuning:
        table = pd.DataFrame(tuning)
        ax2.barh(table.campaign, table.mean_gap, color=["0.6"] * (len(table) - 1) + ["C0"])
        ax2.invert_yaxis()
        ax2.set(xlabel="mean development gap (%)", title="Tuning, one change at a time")
    fig.tight_layout()
    plt.show()
    convergence = TEST_CAMPAIGN / "alns_convergence.png"
    if convergence.is_file():
        from IPython.display import Image
        display(Image(filename=str(convergence), width=620))

### 15.5 · Interpretation (held-out campaign `test-001`)

* **Feasibility and fairness.** All 360 attempts were independently feasible and finished inside the hard limit; the configuration was frozen before any test instance was run, and the solver never sees reference routes.
* **Quality.** Equal-weight mean gap 0.654 % to published BKS, worst single run 2.364 %, no run above 7 % — the shared evaluator's assessment is **PASS** for quality. BKS are best-known, not proven optimal, and no lower bound was computed.
* **Scalability.** The mean gap grows from about 0.3 % at 100–200 customers to 1.2 % at 1,000, while iterations per second fall roughly in proportion to size. The convergence curves are still descending at the deadline for 600–1,000 customers: these sizes are limited by time, not stuck.
* **Weak spots.** Random and mixed geography (R1, R2, RC1) at 600–1,000 customers. Candidate improvements: string cross-exchange, SWAP\*, penalised (time-warp) search, cooperative multi-core search or longer budgets.
* **Not yet done.** PyVRP baseline at equal budgets, ILS and HGS candidates, an agreed project runtime budget and tutor review.

## 16 · References

1. Solomon, M. M. (1987). Algorithms for the vehicle routing and scheduling problems with time window constraints. *Operations Research*, 35(2), 254–265.
2. Gehring, H., & Homberger, J. (1999). A parallel hybrid evolutionary metaheuristic for the vehicle routing problem with time windows. *Proceedings of EURO-GEN99*.
3. [PyVRP/Instances VRPTW collection](https://github.com/PyVRP/Instances/tree/main/VRPTW) — inputs, published reference routes, and evaluation convention.
4. [VRPLIB Python reader](https://github.com/PyVRP/VRPLIB) — parser and solution-file format.
5. Ropke, S., & Pisinger, D. (2006). An adaptive large neighborhood search heuristic for the pickup and delivery problem with time windows. *Transportation Science*, 40(4), 455–472.
6. Pisinger, D., & Ropke, S. (2007). A general heuristic for vehicle routing problems. *Computers & Operations Research*, 34(8), 2403–2435.
7. Shaw, P. (1998). Using constraint programming and local search methods to solve vehicle routing problems. *CP-98*, LNCS 1520, 417–431.
8. Christiaens, J., & Vanden Berghe, G. (2020). Slack induction by string removals for vehicle routing problems. *Transportation Science*, 54(2), 417–433.
9. Vidal, T., Crainic, T. G., Gendreau, M., & Prins, C. (2013). A hybrid genetic algorithm with adaptive diversity management for a large class of vehicle routing problems with time-windows. *Computers & Operations Research*, 40(1), 475–489.
10. Toth, P., & Vigo, D. (2003). The granular tabu search and its application to the vehicle-routing problem. *INFORMS Journal on Computing*, 15(4), 333–346.
11. Hansen, P., & Mladenović, N. (2001). Variable neighborhood search: Principles and applications. *European Journal of Operational Research*, 130(3), 449–467.

Published reference routes are validation inputs. They are not routes discovered by this notebook.

---
### Verification boundary

The live cells report synthetic test results and either a complete 356-reference audit or **NOT RUN**. Strict mode fails when required files are absent. All saved results belong to the timestamped run, package versions, collection fingerprint and per-file hashes above; rerun for fresh evidence. These checks validate supplied route feasibility/cost and small-equation examples, not solver quality or optimality. Sections 12–15 run a short live ALNS + VND demonstration on development instances and load the committed campaigns; the 360-attempt held-out campaign itself is executed with `scripts/run_alns.py`, not during Run All. No final method is selected; tutor review remains pending.
